# SMAC Reporting Layer — SET 1 Tables

**Migration**: `reporting_layer.sac_prod_seafarer_public.*` → `reporting_layer.smac_prod.*`

## Tables in this Notebook

| # | Table | SAC Rows | SMAC Rows | Status |
|---|-------|----------|-----------|--------|
| 1 | `revised_base_view` | 6,838,743 | 3,190,731 | ✅ |
| 2 | `revised_relief_view` | 20,724 | 20,769 | ✅ |
| 3 | `planner_view` | 78,177 | 78,604 | ✅ |
| 4 | `add_digital_appraisal_view` | 26,485 | 28,706 | ✅ |
| 5 | `appraisal_performance` | 576 | 547 | ✅ |
| 6 | `digital_appraisal_view` | 2,637 | 2,788 | ✅ |
| 7 | `inactive_seafarers` | 10,439 | 13,970 | ✅ |

**Run order**: Execute cells top-to-bottom. `planner_view` depends on `revised_base_view` + `revised_relief_view`. `digital_appraisal_view` depends on `appraisal_performance`.


In [0]:
%sql
-- =============================================================================
-- SMAC REVISED BASE VIEW - Full 1:1 Rewrite of SAC Logic
-- Target: reporting_layer.smac_prod.revised_base_view
-- Source: SMAC curated_db views (pre-filtered by _fivetran_active = TRUE)
--
-- ARCHITECTURE (mirrors SAC exactly):
--   1. Inner query: ALL seafarers × ALL sea_experiences (not just latest)
--   2. GEN_MONTH: Calendar join producing monthly rows per experience
--   3. C2 subquery: COMPANY_STATUS, SYNERGY_JOINING_DATE, FIRST/LATEST rank/company
--   4. Dedup CTE: ROW_NUMBER to eliminate duplicates
--
-- NOTE: IDs are now UUID strings (target table schema may need ALTER)
--       appraisals_data now uses SMAC view (reporting_layer.smac_prod.appraisals_data)
--       Direct UUID join on seafarer_id — no SAC bridge needed
-- =============================================================================

-- Using CREATE OR REPLACE to handle schema changes (INT → STRING for UUID fields)
CREATE OR REPLACE TABLE reporting_layer.smac_prod.revised_base_view
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
)
AS

SELECT DISTINCT
  SEAFARER_ID, FIRST_NAME, MIDDLE_NAME, LAST_NAME, SEAFARER_NAME, USER_ID,
  AHOY_STATUS, CREW_CODE, OLD_CREW_CODE, CURRENT_STATUS, SEAFARER_TYPE,
  ANNIVERSARY_DATE, RANK_ID, GENDER_NAME, DATE_OF_BIRTH, AGE, AGE_CATEGORY,
  CREATED_AT, PROFILE_STATUS, ONBOARD_SAILING_STATUS, AVAILABILITY_DATE,
  AVAILABILITY_MONTH, CDC_NUMBER, APPRAISAL_LINK, DOCUMENTS_LINK,
  SEA_EXPERIENCE_LINK, SEAFARER_PROFILE_LINK, CURRENT_RANK_NAME,
  NATIONALITY_NAME, `Last DOC/Contract Company`, CONTACT_NUMBER,
  EMERGENCY_CONTACT_NUMBER, EMAIL_ID, ADDRESS_TYPE, NEW_CONTACT_TYPE,
  PRIMARY_ADDRESS, CITY, PIN_CODE, NEAREST_AIRPORT, STATE, COUNTRY,
  SEA_EXPERIENCE_ID, SIGN_ON_DATE, SIGN_OFF_DATE, CONTRACT_ID,
  ACTIVE_CONTRACT, SAC_CONTRACT, VERIFIED_BY_ID, IS_VERIFIED,
  VERIFIED_BY_NAME, VERIFIED_ON, SIGN_OFF_REASON, RANK_NAME_SE,
  IMO_NUMBER, VESSEL_NAME, VESSEL_ID, SHIP_MANAGEMENT_COMPANY_NAME,
  PORT_OF_REGISTRY_NAME, SHIP_MANAGEMENT_COMPANY_ID, VESSEL_CATEGORY_NAME,
  CAPACITY, DWT, DUAL_FUEL, MAKE_NAME, MODEL_NAME, OUTPUT_POWER, GRT,
  EXPERIENCE_IN_DAYS, EXPERIENCE_IN_MONTHS, EXPERIENCE_IN_MONTHS_ROUNDOFF,
  EXPERIENCE_IN_YEAR, IS_SYNERGY_EXPERIANCE, POD_NAME, FROM_DATE, TO_DATE,
  STATUS, NEED_OF_APPRAISAL, APPRAISALS_RANK_NAME, APPRAISALS_VESSEL_NAME,
  APPRAISALS_VESSEL_CATEGORY_NAME, APPRAISAL_DATE, IS_MANUAL,
  CONTRACT_END_DATE, CONTRACT_START_DATE, TO_PORT_NAME, FROM_PORT_NAME,
  APPRAISAL_STATUS, SYNERGY_COMPANY, RECRUITMENT_COMPANY,
  TENTITIVE_SIGN_OFF_DATE, CONTRACT_STATUS, AGENT_NAME, POSITION_NAME,
  POSITION_RANK_ID, DATE_OF_TERMINATION, REMARK, REMARK_TYPE, INACTIVE_TYPE,
  UPDATED_AT, AVAILABILITY_REMARKS, `Overdue by / Days left`,
  LATEST_CONTRACT_END_DATE, LATEST_SIGN_OFF_DATE, LATEST_SIGN_ON_DATE,
  MONTHS, LATEST_DATE_1, DATE, COMPANY_STATUS, SYNERGY_JOINING_DATE,
  SECOND_LATEST_RANK, FIRST_RANK, FIRST_COMPANY, LATEST_COMPANY,
  VESSEL_FLEET_TYPE, RANK_LEVEL, `Rank Category`, POD_VESSEL_NAME,
  VESSEL_CODE, VESSEL_SUB_CATEGORY
FROM (
  WITH CTE AS (
    SELECT DISTINCT *, 
      ROW_NUMBER() OVER (
        PARTITION BY
          SEAFARER_ID, SEA_EXPERIENCE_ID, MONTHS,
          CREW_CODE, SIGN_ON_DATE, SIGN_OFF_DATE,
          FROM_DATE, TO_DATE, STATUS, NEED_OF_APPRAISAL,
          CONTRACT_STATUS, CONTRACT_END_DATE, CONTRACT_START_DATE,
          VESSEL_NAME, RANK_NAME_SE, SHIP_MANAGEMENT_COMPANY_NAME,
          REMARK, REMARK_TYPE, INACTIVE_TYPE, DATE_OF_TERMINATION,
          EXPERIENCE_IN_DAYS, ONBOARD_SAILING_STATUS, PROFILE_STATUS
        ORDER BY SEAFARER_ID
      ) AS RANK_
    FROM (
      SELECT
        C1.SEAFARER_ID, C1.FIRST_NAME, C1.MIDDLE_NAME, C1.LAST_NAME,
        C1.SEAFARER_NAME, C1.USER_ID, C1.AHOY_STATUS, C1.CREW_CODE,
        C1.OLD_CREW_CODE, C1.CURRENT_STATUS, C1.SEAFARER_TYPE,
        C1.ANNIVERSARY_DATE, C1.RANK_ID, C1.GENDER_NAME, C1.DATE_OF_BIRTH,
        C1.AGE, C1.AGE_CATEGORY, C1.CREATED_AT, C1.PROFILE_STATUS,
        C1.ONBOARD_SAILING_STATUS, C1.AVAILABILITY_DATE, C1.AVAILABILITY_MONTH,
        C1.CDC_NUMBER, C1.APPRAISAL_LINK, C1.DOCUMENTS_LINK,
        C1.SEA_EXPERIENCE_LINK, C1.SEAFARER_PROFILE_LINK, C1.CURRENT_RANK_NAME,
        C1.NATIONALITY_NAME, C1.`Last DOC/Contract Company`, C1.CONTACT_NUMBER,
        C1.EMERGENCY_CONTACT_NUMBER, C1.EMAIL_ID, C1.ADDRESS_TYPE,
        C1.NEW_CONTACT_TYPE, C1.PRIMARY_ADDRESS, C1.CITY, C1.PIN_CODE,
        C1.NEAREST_AIRPORT, C1.STATE, C1.COUNTRY, C1.SEA_EXPERIENCE_ID,
        C1.SIGN_ON_DATE, C1.SIGN_OFF_DATE, C1.CONTRACT_ID, C1.ACTIVE_CONTRACT,
        C1.SAC_CONTRACT, C1.VERIFIED_BY_ID, C1.IS_VERIFIED, C1.VERIFIED_BY_NAME,
        C1.VERIFIED_ON, C1.SIGN_OFF_REASON, C1.RANK_NAME_SE, C1.IMO_NUMBER,
        C1.VESSEL_NAME, C1.VESSEL_ID, C1.SHIP_MANAGEMENT_COMPANY_NAME,
        C1.PORT_OF_REGISTRY_NAME, C1.SHIP_MANAGEMENT_COMPANY_ID,
        C1.VESSEL_CATEGORY_NAME, C1.CAPACITY, C1.DWT, C1.DUAL_FUEL,
        C1.MAKE_NAME, C1.MODEL_NAME, C1.OUTPUT_POWER, C1.GRT,
        C1.EXPERIENCE_IN_DAYS, C1.EXPERIENCE_IN_MONTHS,
        C1.EXPERIENCE_IN_MONTHS_ROUNDOFF, C1.EXPERIENCE_IN_YEAR,
        C1.IS_SYNERGY_EXPERIANCE, C1.POD_NAME, C1.FROM_DATE, C1.TO_DATE,
        C1.STATUS, C1.NEED_OF_APPRAISAL, C1.APPRAISALS_RANK_NAME,
        C1.APPRAISALS_VESSEL_NAME, C1.APPRAISALS_VESSEL_CATEGORY_NAME,
        C1.APPRAISAL_DATE, C1.IS_MANUAL, C1.CONTRACT_END_DATE,
        C1.CONTRACT_START_DATE, C1.TO_PORT_NAME, C1.FROM_PORT_NAME,
        C1.APPRAISAL_STATUS, C1.SYNERGY_COMPANY, C1.RECRUITMENT_COMPANY,
        C1.TENTITIVE_SIGN_OFF_DATE, C1.CONTRACT_STATUS, C1.AGENT_NAME,
        C1.POSITION_NAME, C1.POSITION_RANK_ID, C1.DATE_OF_TERMINATION,
        C1.REMARK, C1.REMARK_TYPE, C1.INACTIVE_TYPE, C1.UPDATED_AT,
        C1.AVAILABILITY_REMARKS,
        C1.POD_VESSEL_NAME,
        C1.VESSEL_CODE,
        C1.VESSEL_SUB_CATEGORY,
        C1.`Overdue by / Days left`,
        C1.LATEST_CONTRACT_END_DATE, C1.LATEST_SIGN_OFF_DATE,
        C1.LATEST_SIGN_ON_DATE, C1.MONTHS,
        -- LATEST_DATE_1 & DATE
        CASE
          WHEN C1.LATEST_DATE IS NULL AND C1.LATEST_SIGN_ON_DATE = C1.SIGN_ON_DATE
          THEN C1.LATEST_SIGN_OFF_DATE
        END AS LATEST_DATE_1,
        COALESCE(C1.LATEST_DATE, LATEST_DATE_1) AS DATE,
        -- C2 derived columns
        C2.COMPANY_STATUS,
        C2.SYNERGY_JOINING_DATE,
        C2.SECOND_LATEST_RANK,
        C2.FIRST_RANK,
        C2.FIRST_COMPANY,
        C2.LATEST_COMPANY,
        -- VESSEL_FLEET_TYPE classification
        CASE
          WHEN UPPER(C1.VESSEL_CATEGORY_NAME) IN (
            'BULK CARRIER','CONTAINER','GEN CARGO / MULTI-PURPOSE VESSEL',
            'CAR CARRIER / RO-RO','CEMENT CARRIER','LOG CARRIER',
            'REEFER CARGO','HEAVY LIFT/PROJECT CARGO','WOODCHIP CARRIER','OBO CARRIER'
          ) THEN 'DRY'
          WHEN UPPER(C1.VESSEL_CATEGORY_NAME) IN (
            'OIL TANKER','CHEM/OIL PROD TANKER','ASPHALT / BITUMEN TANKER',
            'LPG CARRIER (REFRI)','CHEMICAL TANKER','LNG CARRIER',
            'LPG CARRIER (PRESS)','SUPPLY /OFFSHORE / TUG BOAT / AHTS',
            'GAS TANKER','OIL/PROD BUNKER BARGE','CHEM/PROD TANKER'
          ) THEN 'WET'
          ELSE NULL
        END AS VESSEL_FLEET_TYPE,
        -- RANK_LEVEL classification
        CASE
          WHEN C1.CURRENT_RANK_NAME IN ('Deck Cadet','Engine Cadet','Electrical Cadet') THEN 'Cadet'
          WHEN C1.CURRENT_RANK_NAME IN ('Master','Chief Officer','Chief Engineer','Second Engineer') THEN 'Management'
          WHEN C1.CURRENT_RANK_NAME IN ('Third Engineer','Third Officer','Second Officer','Fourth Engineer','Electro Technical Officer','Electrical Officer','Junior Fourth Engineer','Gas Engineer','Junior Third Officer') THEN 'Operational'
          WHEN C1.CURRENT_RANK_NAME IN ('Fitter','Pumpman','Able Bodied Seaman','Oiler','Ordinary Seaman','Chief Cook','Bosun','General Steward','Wiper','Crew') THEN 'Support'
          WHEN C1.CURRENT_RANK_NAME IN ('Trainee Electrical Officer','Trainee Wiper','Trainee Seaman','Trainee Fitter','Trainee General Steward') THEN 'Trainee'
        END AS RANK_LEVEL,
        -- Rank Category classification
        CASE
          WHEN C1.CURRENT_RANK_NAME IN ('Master','Chief Officer','Chief Engineer','Second Engineer') THEN 'Top 4 Rank'
          WHEN C1.CURRENT_RANK_NAME IN ('Additional 3rd Officer','Additional Master','Additional Officer','Deck Cadet','Deck Fitter','Junior Third Officer','Second Officer','Sr Deck cadet','Third Officer','Trainee Master','DNS EXAM','Junior Watchkeeping Officer','Anchor Handler','Additional Second Engineer','Assistant Engineer','Electrical Cadet','Electrical Officer','Electrician','Electro Technical Officer','Engine Cadet','Fourth Engineer','Gas Engineer','Junior Electrical Officer','Junior Engineer','Junior Fourth Engineer','Junior Gas Engineer','SENIOR ELECTRICAL OFFICER','Third Engineer','Trainee Electrical Officer','Trainee Gas Engineer','Trainee Marine Engineer','Trainee Radio Officer','GME EXAM','Electrical Engineer','Deck Girl') THEN 'Officer'
          WHEN C1.CURRENT_RANK_NAME IN ('Able Bodied Seaman','Bosun','Chief Cook','General Steward','Messboy','Messman','Ordinary Seaman','Trainee General Steward','Trainee Ordinary seaman','Trainee Wiper','Wiper','Wiper 1','Crew','Deck Fitter','Fitter','Motorman','Oiler','Oiler 1','Pumpman','Trainee Pumpman','Trainee Seaman','RPFW(Repair Fitter Welder)','Traine Fitter','Trainee Fitter','Electro Technical Rating','Second Cook') THEN 'Rating'
          ELSE NULL
        END AS `Rank Category`
      FROM (
        -- ============================
        -- SOURCE_TABLE + GEN_MONTH
        -- ============================
        WITH SOURCE_TABLE AS (
          SELECT * FROM (
            SELECT
              X.*,
              -- LATEST_DATE logic
              CASE
                WHEN X.SEAFARER_TYPE = 'Internal Seafarers' AND X.ONBOARD_SAILING_STATUS = 'Onboard' AND X.SIGN_ON_DATE = Y.SIGN_ON_DATE THEN Y.CONTRACT_END_DATE
                WHEN X.SEAFARER_TYPE = 'Internal Seafarers' AND X.ONBOARD_SAILING_STATUS = 'Onboard' AND X.SIGN_ON_DATE = Y.SIGN_ON_DATE AND X.CONTRACT_END_DATE IS NULL THEN Y.SIGN_OFF_DATE
                WHEN X.SEAFARER_TYPE IN ('Internal Seafarers','External Seafarers') AND X.ONBOARD_SAILING_STATUS = 'Onleave' AND X.SIGN_ON_DATE = Y.SIGN_ON_DATE THEN (CASE WHEN Y.SIGN_OFF_DATE IS NULL THEN Y.CONTRACT_END_DATE ELSE Y.SIGN_OFF_DATE END)
                WHEN X.SEAFARER_TYPE IN ('Internal Seafarers','External Seafarers') AND X.ONBOARD_SAILING_STATUS = 'No Past Records' THEN to_timestamp('1999-12-31 00:00:00.000')
                WHEN X.SEAFARER_TYPE IN ('Internal Seafarers','External Seafarers') AND Y.SIGN_OFF_DATE IS NULL AND Y.CONTRACT_END_DATE IS NULL THEN to_timestamp('1999-12-31 00:00:00.000')
                WHEN X.SEAFARER_TYPE IN ('Internal Seafarers','External Seafarers') AND X.CONTRACT_END_DATE = Y.CONTRACT_END_DATE AND Y.SIGN_OFF_DATE IS NULL AND Y.CONTRACT_END_DATE IS NOT NULL THEN Y.CONTRACT_END_DATE
                WHEN X.SEAFARER_TYPE IN ('Internal Seafarers','External Seafarers') AND X.SIGN_OFF_DATE = Y.SIGN_OFF_DATE AND Y.SIGN_OFF_DATE IS NOT NULL AND Y.CONTRACT_END_DATE IS NULL THEN Y.SIGN_OFF_DATE
              END AS LATEST_DATE,
              CONCAT(CAST(datediff(CAST(X.CONTRACT_END_DATE AS DATE), current_date()) AS STRING), ' ', 'DAYS') AS `Overdue by / Days left`,
              Y.CONTRACT_END_DATE AS LATEST_CONTRACT_END_DATE,
              Y.SIGN_OFF_DATE AS LATEST_SIGN_OFF_DATE,
              Y.SIGN_ON_DATE AS LATEST_SIGN_ON_DATE
            FROM (
              -- ============================
              -- MAIN INNER QUERY (X)
              -- All seafarers × All sea experiences
              -- ============================
              SELECT
                B.id AS SEAFARER_ID,
                B.first_name AS FIRST_NAME,
                B.middle_name AS MIDDLE_NAME,
                B.last_name AS LAST_NAME,
                CONCAT(COALESCE(B.first_name,' '),' ',COALESCE(B.middle_name,' '),' ',COALESCE(B.last_name,' ')) AS SEAFARER_NAME,
                B.id AS USER_ID,
                CASE WHEN B.identity_profile_id IS NOT NULL THEN 'Ahoy Installed' ELSE 'Ahoy Not Installed' END AS AHOY_STATUS,
                B.crew_code AS CREW_CODE,
                B.old_crew_code AS OLD_CREW_CODE,
                -- CURRENT_STATUS from profile_states
                PS.name AS CURRENT_STATUS,
                -- SEAFARER_TYPE
                CASE
                  WHEN UPPER(PS.code) IN ('REGISTERED','APPLIED','SELECTED') THEN 'External Seafarers'
                  ELSE 'Internal Seafarers'
                END AS SEAFARER_TYPE,
                SP.anniversary_date AS ANNIVERSARY_DATE,
                B.rank_id AS RANK_ID,
                -- GENDER from genders lookup
                COALESCE(GEN.name, 'Unknown') AS GENDER_NAME,
                B.date_of_birth AS DATE_OF_BIRTH,
                CAST(datediff(YEAR, CAST(B.date_of_birth AS DATE), CAST(current_date() AS DATE)) AS INT) AS AGE,
                CASE
                  WHEN AGE < 30 THEN '<30'
                  WHEN AGE BETWEEN 30 AND 49 THEN '30 - 49'
                  WHEN AGE > 50 THEN '50+'
                END AS AGE_CATEGORY,
                B.created_at AS CREATED_AT,
                -- PROFILE_STATUS from seafarer_profile_statuses
                CASE WHEN PST.code = 'ACTIVE' THEN 'Active Seafarer' ELSE 'Inactive Seafarer' END AS PROFILE_STATUS,
                -- ONBOARD_SAILING_STATUS
                CASE
                  WHEN PST.code = 'ACTIVE' AND PS.code = 'SIGNON' THEN 'Onboard'
                  WHEN J.sign_on_date IS NULL AND J.sign_off_date IS NULL THEN 'No Past Records'
                  ELSE 'Onleave'
                END AS ONBOARD_SAILING_STATUS,
                B.availability_date AS AVAILABILITY_DATE,
                month(CAST(B.availability_date AS DATE)) AS AVAILABILITY_MONTH,
                B.cdc_number AS CDC_NUMBER,
                -- Links (using SMAC URLs)
                CONCAT('https://crewing.synergymarine.in/crewing/seafarer/details/', B.id, '/appraisals') AS APPRAISAL_LINK,
                CONCAT('https://crewing.synergymarine.in/crewing/seafarer/details/', B.id, '/documents') AS DOCUMENTS_LINK,
                CONCAT('https://crewing.synergymarine.in/crewing/seafarer/details/', B.id, '/sea-experience') AS SEA_EXPERIENCE_LINK,
                CONCAT('https://crewing.synergymarine.in/crewing/seafarer/details/', B.id, '/personal') AS SEAFARER_PROFILE_LINK,
                -- Rank & Nationality names
                A.name AS CURRENT_RANK_NAME,
                C.name AS NATIONALITY_NAME,
                D.name AS `Last DOC/Contract Company`,
                -- Contact (absorbed into seafarers in SMAC)
                B.phone AS CONTACT_NUMBER,
                CAST(NULL AS STRING) AS EMERGENCY_CONTACT_NUMBER,  -- Not available in SMAC
                B.email AS EMAIL_ID,
                'PERMANENT ADDRESS' AS ADDRESS_TYPE,
                '1' AS NEW_CONTACT_TYPE,
                -- Address from seafarer_profile.primary_address (JSON)
                get_json_object(SP.primary_address, '$.address') AS PRIMARY_ADDRESS,
                get_json_object(SP.primary_address, '$.city') AS CITY,
                get_json_object(SP.primary_address, '$.pinCode') AS PIN_CODE,
                APT.name AS NEAREST_AIRPORT,
                COALESCE(ST.name, ST_ADDR.name) AS STATE,
                CTR.name AS COUNTRY,
                -- Sea Experience columns (ALL experiences, not just latest)
                J.id AS SEA_EXPERIENCE_ID,
                J.sign_on_date AS SIGN_ON_DATE,
                J.sign_off_date AS SIGN_OFF_DATE,
                COALESCE(J.contract_agreement_id, M_FB.id) AS CONTRACT_ID,
                J.active_contract AS ACTIVE_CONTRACT,
                CAST(NULL AS BOOLEAN) AS SAC_CONTRACT,  -- SAC-specific, no SMAC equivalent
                J.verified_by_id AS VERIFIED_BY_ID,
                J.is_verified AS IS_VERIFIED,
                CASE
                  WHEN J.verified_by_id IS NULL THEN NULL
                  WHEN VERIFIER.first_name IS NULL AND VERIFIER.last_name IS NULL THEN 'System'
                  ELSE TRIM(CONCAT(COALESCE(VERIFIER.first_name, ''), ' ', COALESCE(VERIFIER.last_name, '')))
                END AS VERIFIED_BY_NAME,
                J.verified_at AS VERIFIED_ON,
                K.name AS SIGN_OFF_REASON,
                L.name AS RANK_NAME_SE,
                J.imo_number AS IMO_NUMBER,
                COALESCE(J.vessel_name, 'Others') AS VESSEL_NAME,
                J.vessel_id AS VESSEL_ID,
                -- Ship Management Company
                COALESCE(
                  CASE
                    WHEN J.doc_holder_company_id IS NULL THEN J.external_company_name
                    ELSE UPPER(COMP_J.name)
                  END,
                  'Other'
                ) AS SHIP_MANAGEMENT_COMPANY_NAME,
                POR.name AS PORT_OF_REGISTRY_NAME,
                J.doc_holder_company_id AS SHIP_MANAGEMENT_COMPANY_ID,
                VCAT.name AS VESSEL_CATEGORY_NAME,
                -- Capacity & Engine (flattened from JSON in SMAC)
                CONCAT(COALESCE(get_json_object(J.cargo_capacity_info, '$.capacity'), ''), ' ', COALESCE(get_json_object(J.cargo_capacity_info, '$.capacity_unit'), '')) AS CAPACITY,
                CAST(J.dwt AS INT) AS DWT,
                CASE
                  WHEN get_json_object(J.engine_specifications, '$.DualFuel') = 'true' THEN 'YES'
                  ELSE 'NO'
                END AS DUAL_FUEL,
                get_json_object(J.engine_specifications, '$.EngineMakeName') AS MAKE_NAME,
                get_json_object(J.engine_specifications, '$.EngineModelName') AS MODEL_NAME,
                CONCAT(
                  COALESCE(get_json_object(J.engine_specifications, '$.OutputPower'), ''),
                  ' ',
                  COALESCE(UPPER(get_json_object(J.engine_specifications, '$.OutputPowerUnit')), '')
                ) AS OUTPUT_POWER,
                CAST(J.grt AS STRING) AS GRT,
                -- Experience calculations
                CASE
                  WHEN ONBOARD_SAILING_STATUS = 'Onboard' THEN datediff(current_date(), CAST(J.sign_on_date AS DATE))
                  ELSE COALESCE(J.duration_days, 0)
                END AS EXPERIENCE_IN_DAYS,
                CAST(EXPERIENCE_IN_DAYS AS DOUBLE) / 30 AS EXPERIENCE_IN_MONTHS,
                ROUND(CAST(EXPERIENCE_IN_DAYS AS DOUBLE) / 30) AS EXPERIENCE_IN_MONTHS_ROUNDOFF,
                ROUND((CAST(EXPERIENCE_IN_DAYS AS DOUBLE) / 30) / 12, 1) AS EXPERIENCE_IN_YEAR,
                J.is_inhouse_experience AS IS_SYNERGY_EXPERIANCE,
                -- POD
                POD.POD AS POD_NAME,
                -- Appraisals (from existing view)
                N.FROM_DATE,
                N.TO_DATE,
                N.STATUS,
                N.NEED_OF_APPRAISAL,
                N.APPRAISALS_RANK_NAME,
                N.APPRAISALS_VESSEL_NAME,
                N.APPRAISALS_VESSEL_CATEGORY_NAME,
                N.APPRAISAL_DATE,
                N.IS_MANUAL,
                -- Contract (primary: contract_agreement_id; fallback: seafarer_contracts + agreement when FK null)
                -- Date precedence when FK null: parent seafarer_contract (SAC vessel_contracts) before agreement addendum
                COALESCE(M.end_date, SC_FB.end_date, M_FB.end_date) AS CONTRACT_END_DATE,
                COALESCE(M.start_date, SC_FB.start_date, M_FB.start_date) AS CONTRACT_START_DATE,
                COALESCE(O.name, 'Unknown') AS TO_PORT_NAME,
                COALESCE(P.name, 'Unknown') AS FROM_PORT_NAME,
                CASE WHEN N.FROM_DATE IS NOT NULL THEN 'Completed' ELSE 'Not Completed' END AS APPRAISAL_STATUS,
                -- Synergy Company flag
                CASE
                  WHEN J.doc_holder_company_id IS NULL THEN FALSE
                  ELSE COALESCE(COMP_J.is_inhouse_company, FALSE)
                END AS SYNERGY_COMPANY,
                REC.name AS RECRUITMENT_COMPANY,
                COALESCE(J.sign_off_date, M.end_date, SC_FB.end_date, M_FB.end_date) AS TENTITIVE_SIGN_OFF_DATE,
                CASE
                  WHEN COALESCE(M.id, M_FB.id, SC_FB.id) IS NULL THEN NULL
                  WHEN COALESCE(M.agreement_status, M_FB.agreement_status) IN ('Void', 'Cancelled', 'Terminated') THEN 'Void'
                  WHEN SC_FB.status = 'Inactive' THEN 'Closed'
                  WHEN COALESCE(M.agreement_status, M_FB.agreement_status) = 'Signed' AND J.sign_off_date IS NULL THEN 'InForce'
                  WHEN J.sign_off_date IS NULL AND COALESCE(M.end_date, SC_FB.end_date, M_FB.end_date) >= CURRENT_DATE THEN 'InForce'
                  WHEN J.sign_off_date IS NULL THEN 'Active'
                  WHEN COALESCE(M.end_date, SC_FB.end_date, M_FB.end_date) < CURRENT_DATE THEN 'Closed'
                  WHEN COALESCE(M.agreement_status, M_FB.agreement_status) = 'Signed' THEN 'Signed'
                  ELSE 'Closed'
                END AS CONTRACT_STATUS,
                AGT.name AS AGENT_NAME,
                Z.name AS POSITION_NAME,
                Z.rank_id AS POSITION_RANK_ID,
                -- Remarks
                R.date_of_termination AS DATE_OF_TERMINATION,
                R.remark AS REMARK,
                R.remark_type AS REMARK_TYPE,
                R.Inactive_Type AS INACTIVE_TYPE,
                R.UPDATED_AT,
                AR.name AS AVAILABILITY_REMARKS,
                POD.vessel_name AS POD_VESSEL_NAME,
                VR.code AS VESSEL_CODE,
                VSC.name AS VESSEL_SUB_CATEGORY
              FROM (
                SELECT * FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarers
                WHERE deleted_at IS NULL
              ) B
              -- Profile State
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_crewing.profile_states PS
                ON B.profile_state_id = PS.id
              -- Profile Status
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_crewing.seafarer_profile_statuses PST
                ON B.profile_status_id = PST.id
              -- Gender
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.genders GEN
                ON B.gender_id = GEN.id
              -- Seafarer Profile (address, anniversary)
              LEFT JOIN curated_db.db_smac_prod_navitasai_crewing_public.seafarer_profile SP
                ON B.id = SP.seafarer_id AND SP.deleted_at IS NULL
              -- Rank
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.ranks A
                ON A.id = B.rank_id
              -- Nationality
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.nationalities C
                ON C.id = B.nationality_id
              -- Present DOC Company
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.companies D
                ON D.id = B.present_doc_company_id
              -- Recruitment Company
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.companies REC
                ON REC.id = B.recruitment_company_id
              -- State (from profile address - using B.state_id on seafarers)
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.states ST
                ON ST.id = B.state_id
              -- Country
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.countries CTR
                ON CTR.id = B.country_id
              -- Nearest Airport (from primary_address JSON airportId)
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.airports APT
                ON APT.id = get_json_object(SP.primary_address, '$.airportId')
              -- State from address JSON (fallback when B.state_id is NULL)
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.states ST_ADDR
                ON ST_ADDR.id = get_json_object(SP.primary_address, '$.stateId')
              -- ALL Sea Experiences (not just latest!)
              LEFT JOIN (
                SELECT * FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarer_sea_experiences
                WHERE deleted_at IS NULL
              ) J ON J.seafarer_id = B.id
              -- Verified By (IDP users table)
              LEFT JOIN curated_db.db_smac_prod_navitasai_idp_public.users VERIFIER
                ON VERIFIER.id = J.verified_by_id
              -- Sign Off Reason
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_crewing.sign_off_reasons K
                ON K.id = J.sign_off_reason_id
              -- Rank (sea experience)
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.ranks L
                ON L.id = J.rank_id
              -- Contract agreement (primary FK on sea experience)
              LEFT JOIN curated_db.db_smac_prod_navitasai_crewing_public.contract_agreements M
                ON M.id = J.contract_agreement_id AND M.deleted_at IS NULL
              -- Fallback: best seafarer_contract per experience when contract_agreement_id IS NULL
              -- SAC equivalent: vessel_contracts via sea_experience.CONTRACT_ID (often unset in SMAC source)
              -- Match priority: exact sign_on=start > sign_on in [start,end] > vessel_id + date range
              LEFT JOIN (
                SELECT
                  se.id AS sea_experience_id,
                  sc.id,
                  sc.start_date,
                  sc.end_date,
                  sc.status,
                  ROW_NUMBER() OVER (
                    PARTITION BY se.id
                    ORDER BY
                      CASE
                        WHEN CAST(sc.start_date AS DATE) = CAST(se.sign_on_date AS DATE) THEN 1
                        WHEN CAST(se.sign_on_date AS DATE) BETWEEN CAST(sc.start_date AS DATE)
                          AND CAST(COALESCE(sc.end_date, se.sign_off_date, current_date()) AS DATE) THEN 2
                        WHEN sc.vessel_id IS NOT NULL AND sc.vessel_id = se.vessel_id
                          AND CAST(se.sign_on_date AS DATE) BETWEEN CAST(sc.start_date AS DATE)
                          AND CAST(COALESCE(sc.end_date, se.sign_off_date, current_date()) AS DATE) THEN 3
                        ELSE 4
                      END,
                      sc.start_date DESC
                  ) AS rn
                FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarer_sea_experiences se
                INNER JOIN curated_db.db_smac_prod_navitasai_crewing_public.seafarer_contracts sc
                  ON sc.seafarer_id = se.seafarer_id AND sc.deleted_at IS NULL
                  AND (
                    CAST(sc.start_date AS DATE) = CAST(se.sign_on_date AS DATE)
                    OR CAST(se.sign_on_date AS DATE) BETWEEN CAST(sc.start_date AS DATE)
                      AND CAST(COALESCE(sc.end_date, se.sign_off_date, current_date()) AS DATE)
                    OR (sc.vessel_id IS NOT NULL AND sc.vessel_id = se.vessel_id
                      AND CAST(se.sign_on_date AS DATE) BETWEEN CAST(sc.start_date AS DATE)
                      AND CAST(COALESCE(sc.end_date, se.sign_off_date, current_date()) AS DATE))
                  )
                WHERE se.deleted_at IS NULL AND se.contract_agreement_id IS NULL
              ) SC_FB
                ON SC_FB.sea_experience_id = J.id AND SC_FB.rn = 1 AND J.contract_agreement_id IS NULL
              -- Fallback agreement from parent seafarer_contract (prefer Signed > Approved, latest end_date)
              LEFT JOIN (
                SELECT ca.*,
                  ROW_NUMBER() OVER (
                    PARTITION BY ca.contract_id
                    ORDER BY CASE ca.agreement_status WHEN 'Signed' THEN 1 WHEN 'Approved' THEN 2 ELSE 3 END,
                      ca.end_date DESC,
                      ca.updated_at DESC
                  ) AS rn
                FROM curated_db.db_smac_prod_navitasai_crewing_public.contract_agreements ca
                WHERE ca.deleted_at IS NULL
              ) M_FB ON M_FB.contract_id = SC_FB.id AND M_FB.rn = 1 AND J.contract_agreement_id IS NULL
              -- Position
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.positions Z
                ON Z.id = J.position_id
              -- Appraisals (SMAC view - direct UUID join)
              LEFT JOIN reporting_layer.smac_prod.appraisals_data N
                ON N.SEAFARER_ID = B.id
                AND to_date(N.FROM_DATE) BETWEEN to_date(J.sign_on_date) AND to_date(COALESCE(J.sign_off_date, M.end_date, SC_FB.end_date, M_FB.end_date))
                AND J.vessel_name = N.APPRAISALS_VESSEL_NAME
              -- Sign-off Port
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.ports O
                ON O.id = J.sign_off_port_id
              -- Sign-on Port
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.ports P
                ON P.id = J.sign_on_port_id
              -- Company for Synergy check (via doc_holder on sea exp)
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.companies COMP_J
                ON COMP_J.id = J.doc_holder_company_id
              -- Agent
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.agents AGT
                ON AGT.id = B.manning_agent_id
              -- Availability Remarks
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_crewing.availability_remarks AR
                ON AR.id = B.availability_remark_id
              -- Vessel Revisions (for vessel_code) - latest revision per vessel (any status)
              LEFT JOIN (
                SELECT vessel_id, code
                FROM curated_db.db_smac_prod_navitasai_masters_vessel.vessel_revisions
                WHERE deleted_at IS NULL
                QUALIFY ROW_NUMBER() OVER (PARTITION BY vessel_id ORDER BY status ASC, effective_date DESC, created_at DESC) = 1
              ) VR ON VR.vessel_id = J.vessel_id
              -- Vessel Sub Category
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_vessel.sub_categories VSC
                ON VSC.id = J.vessel_sub_category_id
              -- Vessel Category
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_vessel.categories VCAT
                ON VCAT.id = J.vessel_category_id
              -- Port of Registry
              LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.ports POR
                ON POR.id = J.port_of_registry_id
              -- POD (Place of Delivery) via fleet/vessel mapping
              LEFT JOIN (
                SELECT DISTINCT f.name AS POD, V.imo_number, V.name AS vessel_name
                FROM curated_db.db_smac_prod_navitasai_masters_vessel.fleets f
                LEFT JOIN (
                  SELECT * FROM curated_db.db_smac_prod_navitasai_masters_vessel.fld_fleet_vessels
                  WHERE deleted_at IS NULL
                ) fv ON f.id = fv.fleet_id
                LEFT JOIN curated_db.db_smac_prod_navitasai_masters_vessel.vessels V
                  ON V.id = fv.vessel_id
                WHERE f.deleted_at IS NULL
              ) POD ON POD.imo_number = J.imo_number
              -- Remarks (latest per seafarer)
              LEFT JOIN (
                SELECT seafarer_id, UPDATED_AT, date_of_termination, remark, remark_type, Inactive_Type
                FROM (
                  SELECT
                    R.seafarer_id,
                    R.updated_at AS UPDATED_AT,
                    R.date_of_action AS date_of_termination,
                    R.remark_text AS remark,
                    PRT.name AS remark_type,
                    PRR.name AS Inactive_Type,
                    ROW_NUMBER() OVER (PARTITION BY R.seafarer_id ORDER BY R.updated_at DESC) AS RNUM
                  FROM curated_db.db_smac_prod_navitasai_crewing_shore.seafarer_remarks R
                  LEFT JOIN curated_db.db_smac_prod_navitasai_masters_crewing.profile_remark_types PRT
                    ON PRT.id = R.profile_remark_type_id
                  LEFT JOIN curated_db.db_smac_prod_navitasai_masters_crewing.profile_remark_reasons PRR
                    ON PRR.id = R.profile_remark_reason_id
                  WHERE R.deleted_at IS NULL
                ) T WHERE RNUM = 1
              ) R ON R.seafarer_id = B.id
            ) X
            -- Y subquery: Latest dates per seafarer
            LEFT JOIN (
              SELECT
                A.id AS SEAFARER_ID,
                MAX(B.sign_off_date) AS SIGN_OFF_DATE,
                MAX(COALESCE(CA.end_date, SC.end_date, CA_FB.end_date)) AS CONTRACT_END_DATE,
                MAX(B.sign_on_date) AS SIGN_ON_DATE
              FROM (
                SELECT * FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarers
                WHERE deleted_at IS NULL
              ) A
              LEFT JOIN (
                SELECT * FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarer_sea_experiences
                WHERE deleted_at IS NULL
              ) B ON B.seafarer_id = A.id
              LEFT JOIN curated_db.db_smac_prod_navitasai_crewing_public.contract_agreements CA
                ON CA.id = B.contract_agreement_id AND CA.deleted_at IS NULL
              LEFT JOIN (
                SELECT
                  se.id AS sea_experience_id,
                  sc.id,
                  sc.start_date,
                  sc.end_date,
                  ROW_NUMBER() OVER (
                    PARTITION BY se.id
                    ORDER BY
                      CASE
                        WHEN CAST(sc.start_date AS DATE) = CAST(se.sign_on_date AS DATE) THEN 1
                        WHEN CAST(se.sign_on_date AS DATE) BETWEEN CAST(sc.start_date AS DATE)
                          AND CAST(COALESCE(sc.end_date, se.sign_off_date, current_date()) AS DATE) THEN 2
                        WHEN sc.vessel_id IS NOT NULL AND sc.vessel_id = se.vessel_id
                          AND CAST(se.sign_on_date AS DATE) BETWEEN CAST(sc.start_date AS DATE)
                          AND CAST(COALESCE(sc.end_date, se.sign_off_date, current_date()) AS DATE) THEN 3
                        ELSE 4
                      END,
                      sc.start_date DESC
                  ) AS rn
                FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarer_sea_experiences se
                INNER JOIN curated_db.db_smac_prod_navitasai_crewing_public.seafarer_contracts sc
                  ON sc.seafarer_id = se.seafarer_id AND sc.deleted_at IS NULL
                  AND (
                    CAST(sc.start_date AS DATE) = CAST(se.sign_on_date AS DATE)
                    OR CAST(se.sign_on_date AS DATE) BETWEEN CAST(sc.start_date AS DATE)
                      AND CAST(COALESCE(sc.end_date, se.sign_off_date, current_date()) AS DATE)
                    OR (sc.vessel_id IS NOT NULL AND sc.vessel_id = se.vessel_id
                      AND CAST(se.sign_on_date AS DATE) BETWEEN CAST(sc.start_date AS DATE)
                      AND CAST(COALESCE(sc.end_date, se.sign_off_date, current_date()) AS DATE))
                  )
                WHERE se.deleted_at IS NULL AND se.contract_agreement_id IS NULL
              ) SC ON SC.sea_experience_id = B.id AND SC.rn = 1 AND B.contract_agreement_id IS NULL
              LEFT JOIN (
                SELECT ca.*,
                  ROW_NUMBER() OVER (
                    PARTITION BY ca.contract_id
                    ORDER BY CASE ca.agreement_status WHEN 'Signed' THEN 1 WHEN 'Approved' THEN 2 ELSE 3 END,
                      ca.end_date DESC,
                      ca.updated_at DESC
                  ) AS rn
                FROM curated_db.db_smac_prod_navitasai_crewing_public.contract_agreements ca
                WHERE ca.deleted_at IS NULL
              ) CA_FB ON CA_FB.contract_id = SC.id AND CA_FB.rn = 1 AND B.contract_agreement_id IS NULL
              GROUP BY A.id
            ) Y ON X.SEAFARER_ID = Y.SEAFARER_ID
          )
        ),
        -- GEN_MONTH: Calendar table for monthly grain
        GEN_MONTH AS (
          SELECT MONTHS FROM (
            SELECT add_months(to_date('2000-01-01'), month_offset) AS MONTHS
            FROM (SELECT explode(sequence(0, CAST(months_between(current_date(), to_date('2000-01-01')) AS INT))) AS month_offset)
          )
          ORDER BY MONTHS
        )
        SELECT ST.*, GM.MONTHS
        FROM SOURCE_TABLE AS ST
        LEFT JOIN GEN_MONTH AS GM
          ON GM.MONTHS BETWEEN
            make_date(year(ST.SIGN_ON_DATE), month(ST.SIGN_ON_DATE), 1)
          AND
            make_date(
              year(CASE WHEN ST.SIGN_OFF_DATE IS NULL THEN current_date() ELSE ST.SIGN_OFF_DATE END),
              month(CASE WHEN ST.SIGN_OFF_DATE IS NULL THEN current_date() ELSE ST.SIGN_OFF_DATE END),
              1
            )
        ORDER BY CREW_CODE, VESSEL_ID, SEAFARER_ID
      ) C1
      -- ============================
      -- C2: COMPANY STATUS SUBQUERY
      -- ============================
      LEFT JOIN (
        SELECT
          T2.*,
          CASE
            WHEN COUNT(SIGN_ON_DATE) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE) = 1 THEN 'New Hand'
            WHEN UPPER(STATUS) = 'SIGNON' AND UPPER(LAG(DOC_COMPANY) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE)) = 'FALSE' AND UPPER(LAG(SYNERGY_COMPANY) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE)) = 'FALSE' AND FROM_DATE = SIGN_ON_DATE AND RANK = 1 THEN 'New Hand'
            WHEN UPPER(STATUS) = 'SIGNON' AND UPPER(LAG(SYNERGY_COMPANY) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE)) = 'FALSE' AND UPPER(LAG(DOC_COMPANY) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE)) = 'TRUE' AND FROM_DATE = SIGN_ON_DATE AND RANK = 1 THEN 'Ex Hand'
            WHEN UPPER(STATUS) = 'SIGNON' AND UPPER(LAG(SYNERGY_COMPANY) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE)) = 'TRUE' AND FROM_DATE = SIGN_ON_DATE AND RANK = 1 THEN 'Ex Hand'
            WHEN UPPER(STATUS) <> 'SIGNON' AND RANK = 1 AND UPPER(SYNERGY_COMPANY) = 'FALSE' THEN 'New Hand'
            WHEN UPPER(STATUS) <> 'SIGNON' AND RANK = 1 AND UPPER(SYNERGY_COMPANY) = 'TRUE' THEN 'Ex Hand'
            WHEN FROM_DATE IS NULL AND TO_DATE IS NULL THEN 'New Hand'
            WHEN UPPER(STATUS) = 'SIGNON' AND MAX(RANK) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE) = '1' THEN 'New Hand'
            ELSE ''
          END AS COMPANY_STATUS,
          CASE
            WHEN RANK = 1 AND COMPANY_STATUS IN ('New Hand','Ex Hand')
            THEN MIN(CASE WHEN UPPER(SYNERGY_COMPANY) = 'TRUE' THEN SIGN_ON_DATE ELSE NULL END) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE)
            ELSE NULL
          END AS SYNERGY_JOINING_DATE,
          CASE
            WHEN LAG(RANK) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE) = 2 AND RANK = 1
            THEN LAG(POSITION_NAME) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE)
          END AS SECOND_LATEST_RANK,
          CASE
            WHEN MIN(RANK_1) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE) = 1 AND RANK = 1
            THEN LAST_VALUE(POSITION_NAME) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE DESC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
          END AS FIRST_RANK,
          CASE
            WHEN MIN(RANK_1) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE) = 1 AND RANK = 1
            THEN LAST_VALUE(SHIP_MANAGEMENT_COMPANY_NAME) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE DESC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
          END AS FIRST_COMPANY,
          CASE
            WHEN ONBOARD_SAILING_STATUS = 'Onboard' THEN (
              CASE WHEN MIN(RANK_1) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE) = 1 AND RANK = 1
                THEN NTH_VALUE(SHIP_MANAGEMENT_COMPANY_NAME, 2) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE DESC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
              END
            )
            WHEN ONBOARD_SAILING_STATUS = 'Onleave' THEN (
              CASE WHEN MIN(RANK_1) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE) = 1 AND RANK = 1
                THEN FIRST_VALUE(SHIP_MANAGEMENT_COMPANY_NAME) OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE DESC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
              END
            )
            ELSE NULL
          END AS LATEST_COMPANY
        FROM (
          SELECT
            T1.*,
            DENSE_RANK() OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE DESC) AS RANK,
            DENSE_RANK() OVER (PARTITION BY SEAFARER_ID ORDER BY SIGN_ON_DATE ASC) AS RANK_1
          FROM (
            SELECT
              B.id AS SEAFARER_ID,
              B.crew_code,
              MAX(J.sign_on_date) AS SIGN_ON_DATE,
              PS.code AS STATUS,
              J.duration_days AS EXPERIENCE_IN_DAYS,
              J.sign_on_date AS FROM_DATE,
              J.sign_off_date AS TO_DATE,
              J.id AS SEA_EXPERIENCE_ID,
              Z.name AS POSITION_NAME,
              J.external_company_name AS EXTERNAL_COMPANY_NAME,
              COALESCE(J.vessel_name, 'Others') AS VESSEL_NAME,
              COALESCE(
                CASE
                  WHEN J.doc_holder_company_id IS NULL THEN J.external_company_name
                  ELSE UPPER(COMP_Q.name)
                END, 'Other'
              ) AS SHIP_MANAGEMENT_COMPANY_NAME,
              J.doc_holder_company_id AS SHIP_MANAGEMENT_COMPANY_ID,
              CASE
                WHEN J.doc_holder_company_id IS NULL THEN 'FALSE'
                ELSE CAST(COALESCE(COMP_Q.is_inhouse_company, FALSE) AS STRING)
              END AS SYNERGY_COMPANY,
              COMP_Q.id AS SHIP_MANAGEMENT_COMPANY_ID_2,
              -- DOC_COMPANY: check if company has DOC service type
              CASE WHEN DOC_CS.company_id IS NOT NULL THEN 'TRUE' ELSE 'FALSE' END AS DOC_COMPANY,
              PS.name AS CURRENT_STATUS,
              CASE WHEN PST.code = 'ACTIVE' THEN 'Active Seafarer' ELSE 'Inactive Seafarer' END AS PROFILE_STATUS,
              CASE
                WHEN PST.code = 'ACTIVE' AND PS.code = 'SIGNON' THEN 'Onboard'
                WHEN J.sign_on_date IS NULL AND J.sign_off_date IS NULL THEN 'No Past Records'
                ELSE 'Onleave'
              END AS ONBOARD_SAILING_STATUS
            FROM (
              SELECT * FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarers
              WHERE deleted_at IS NULL
            ) B
            LEFT JOIN curated_db.db_smac_prod_navitasai_masters_crewing.profile_states PS
              ON B.profile_state_id = PS.id
            LEFT JOIN curated_db.db_smac_prod_navitasai_masters_crewing.seafarer_profile_statuses PST
              ON B.profile_status_id = PST.id
            LEFT JOIN (
              SELECT * FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarer_sea_experiences
              WHERE deleted_at IS NULL
            ) J ON B.id = J.seafarer_id
            LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.positions Z
              ON Z.id = J.position_id
            LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.companies COMP_Q
              ON COMP_Q.id = J.doc_holder_company_id
            -- DOC Company flag via company_services junction
            LEFT JOIN (
              SELECT DISTINCT company_id
              FROM curated_db.db_smac_prod_navitasai_masters_public.company_services
              WHERE service_type_id = '01963dd1-5f8d-7a3a-b099-11938b981183'
                AND deleted_at IS NULL
            ) DOC_CS ON DOC_CS.company_id = COMP_Q.id
            GROUP BY
              B.id, B.crew_code, PS.code, PS.name, PST.code,
              J.duration_days, J.sign_on_date, J.sign_off_date, J.id,
              Z.name, J.external_company_name, J.vessel_name,
              J.doc_holder_company_id, COMP_Q.name, COMP_Q.id, COMP_Q.is_inhouse_company,
              DOC_CS.company_id
          ) T1
        ) T2
      ) C2 ON C1.SEA_EXPERIENCE_ID = C2.SEA_EXPERIENCE_ID
    )
  )
  SELECT CTE.* EXCEPT (RANK_)
  FROM CTE
  WHERE RANK_ = 1
)

## Section 2: Relief & Planner Views

`revised_relief_view` → `planner_view` (planner depends on both base + relief)

In [0]:
%sql
-- =============================================================================
-- SMAC REVISED_RELIEF_VIEW - Migration from SAC
-- Source: reporting_layer.sac_prod_seafarer_public.REVISED_RELIEF_VIEW
-- Target: reporting_layer.smac_prod.revised_relief_view
--
-- MAPPING:
--   SAC RELIEFS → SMAC seafarer_reliefs (crewing_public)
--   SAC SHORTLISTED_SEAFARERS → SMAC relief_candidates (crewing_shore)
--   SAC VESSEL_CONTRACTS → SMAC seafarer_vessel_assignments (via offsigner_assignment_id)
--   SAC RELIEVING_SEAFARER_ID → SMAC offsigner_id
--   SAC RELIEVER_SEAFARER_ID → SMAC onsigner_id
--   SAC VESSEL_INFO JSON → SMAC flat columns (vessel_name, etc.)
--   SAC GENDER '0'/'1' → SMAC gender_id → genders.name
--   SAC RELIEF_STATE → SMAC relief_states.code (via relief_state_id)
--   SAC REASON → SMAC remarks
--   SAC SIGN_ON_PORT_ID → SMAC joining_place_id
--
-- MAPPED FROM SVA: reliever_travel_state (onsigner SVA), relieving_travel_state (offsigner SVA),
--   flag_documentation_state (INT→string), general_documentation_state (INT→string),
--   documentation_state (derived from flag+general+joining >= 2)
-- NOT MIGRATED (NULL): reliever_sf_status_code (0% in SAC), relieving_sf_status_code (0.58% in SAC),
--   travel_replan_state (0% in SAC)
-- =============================================================================

CREATE OR REPLACE TABLE reporting_layer.smac_prod.revised_relief_view
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
)
AS
SELECT T.*,
  ROW_NUMBER() OVER (PARTITION BY CREW_CODE, SEAFARER_ID, DELAYED_MONTHS ORDER BY SIGN_ON_DATE DESC) AS ROW_NUM
FROM (
  WITH SOURCE_TABLE AS (
    SELECT
      SVA.contract_start_date AS CONTRACT_START_DATE,
      SVA.contract_end_date AS CONTRACT_END_DATE,
      E.cdc_number AS CDC_NUMBER,
      E.crew_code AS CREW_CODE,
      A.sign_on_date AS SIGN_ON_DATE,
      A.sign_off_date AS SIGN_OFF_DATE,
      A.vessel_name AS VESSEL_NAME,
      CAT_A.name AS VESSEL_CATEGORY_NAME,
      L.vessel_name AS PROPOSED_VESSEL_NAME,
      L.sign_off_date AS RELIEVER_SIGN_OFF_DATE,
      -- BULK_TYPE classification (hardcoded vessel names - same as SAC)
      CASE
        WHEN upper(trim(A.vessel_name)) IN (
          'GENCO AQUITAINE','GENCO ARDENNES','GENCO AUVERGNE','GENCO BOURGOGNE','GENCO BRITTANY',
          'GENCO CONSTANTINE','GENCO DEFENDER','GENCO ENDEAVOUR','GENCO HUNTER','GENCO LANGUEDOC',
          'GENCO PICARDY','GENCO PREDATOR','GENCO PYRENEES','GENCO RESOLUTE','GENCO RHONE',
          'GENCO WARRIOR','GENCO TIBERIUS','GENCO TIGER','GENCO AUGUSTUS','GENCO TITUS',
          'GENCO COMMODUS','BALTIC WOLF','GENCO CLAUDIUS','GENCO HADRIAN','GENCO LONDON',
          'BALTIC MANTIS','BALTIC WASP','GENCO LION','BALTIC SCORPION','BALTIC HORNET',
          'GENCO MAXIMUS','BALTIC BEAR','GENCO ENTERPRISE','GENCO MADELEINE','GENCO MAYFLOWER',
          'GENCO CONSTELLATION','GENCO MARY','GENCO LADDEY','GENCO LIBERTY','GENCO COLUMBIA',
          'GENCO WEATHERLY','GENCO MAGIC','GENCO VIGILANT','GENCO FREEDOM','GENCO RANGER'
        ) THEN 'Bulk1'
        WHEN upper(trim(A.vessel_name)) IN (
          'AMIS ACE','AMIS BRAVE','AMIS ELEGANCE','AMIS FORTUNE','AMIS INTEGRITY',
          'AMIS JUSTICE','AMIS KALON','AMIS LEADER','AMIS POWER','AMIS WISDOM I',
          'AMIS WISDOM II','AMIS WISDOM III','BLUE HORIZON','BUNUN KALON','BUNUN ORCHID',
          'BUNUN WISDOM','BUNUN XCEL','BUNUN YOUTH','BUNUN ZEST','CLEAR HORIZON',
          'COREFORTUNE OL','COREOCEAN OL','CORESKY OL','DAIWAN HERO','DAIWAN INFINITY',
          'DAIWAN KALON','DAIWAN WISDOM','ETERNITY SW','FRONTIER BONANZA','GLOBAL FAITH',
          'GOLDEN KIKU','KATAGALAN ACE','KATAGALAN BRAVE','MOONBRIGHT SW','NALUHU',
          'POAVOSA WISDOM','POAVOSA WISDOM III','SAKIZAYA ACE','SAKIZAYA CHAMPION',
          'SAKIZAYA RESPECT','SAKIZAYA XCEL','SCARLET EAGLE','TAOKAS WISDOM','TRANSFORMER OL'
        ) THEN 'Bulk2'
        WHEN upper(trim(A.vessel_name)) IN (
          'OCEAN GLSR','IBERIAN BULKER','AFRICAN BULKER','BERGE NYANGANI','WORLD RUBY',
          'WORLD PRIZE','WORLD DIANA','WORLD VIRTUE','NORD ENERGY','NORD POWER','MESK',
          'ASIAN BULKER','AUSTRALIAN BULKER','ICELAND BULKER','NORD MAGNES','LAVENDER',
          'LOWLANDS BLUE','LOWLANDS DAWN','LOWLANDS FUTURE','LOWLANDS HORIZON','LOWLANDS RISE',
          'LOWLANDS SAGE','LOWLANDS OPAL','LOWLANDS YELLOW','ACE ETERNITY','AN LI','BENITAMOU',
          'GLOBAL HARMONY','MH ADAGIO','NEO','OCCITAN KEY','OCCITAN PAUILLAC','OCCITAN SKY',
          'SATIGNY','KEY JOURNEY','MH ARPEGGIO','NORD FERRUM','K. IRON MOUNTAIN','K RUBY',
          'K VICTORY','K PREMIUM ORE','K CONFIDENCE','IRON PHOENIX','IRON MIRACLE'
        ) THEN 'Bulk3'
        WHEN upper(trim(A.vessel_name)) IN (
          'CARL OLDENDORFF','CHARLOTTE OLDENDORFF','CHRISTINE OLDENDORFF','CONRAD OLDENDORFF',
          'CORA OLDENDORFF','KIM OLDENDORFF','KLARA OLDENDORFF','KNUT OLDENDORFF','EGE-M',
          'PATRICIA OLDENDORFF','PAUL OLDENDORFF','PENELOPE OLDENDORFF','PETER OLDENDORFF',
          'PHILIPP OLDENDORFF','PIA OLDENDORFF','EPICURUS','ETOILE','TRUE NEPTUNE',
          'TRUE CONRAD','TRUE CHAMPION','TAURUS','CL XUCHANG','CL CHANGSHA','CL LUZHOU',
          'CL YANGZHOU','CL GANJIANG','CL LIANYUNGANG','PACIFIC EAST','PACIFIC WEST',
          'PACIFIC NORTH','PACIFIC SOUTH','KM HAKATA','AM OCEAN SILVER','AM OCEAN STAR',
          'VINAYAK','LAMPARD','ZOLA','GH KAHLO','AM UMANG','AM KIRTI','AM TARANG',
          'GCL YAMUNA','GCL SABARMATI','GCL GANGA','GCL TAPI','GCL NARMADA','GCL MAHANADI'
        ) THEN 'Bulk4'
        ELSE 'Others'
      END AS BULK_TYPE,
      F.name AS RANK_NAME,
      COMP.name AS `DOC/CONTRACT_COMPANY`,
      (CAST(SVA.contract_end_date AS DATE) - CURRENT_DATE) || ' DAYS' AS `Overdue by / Days left`,
      E.id AS SEAFARER_ID,
      G.name AS NATIONALITY_NAME,
      V.imo_number AS VESSEL_IMO_NUMBER,
      SVA.contract_id AS VESSEL_CONTRACT_ID,
      -- NOT MIGRATED to SMAC (per playbook line 287 - no equivalent in SMAC)
      CAST(NULL AS STRING) AS RELIEVER_SF_STATUS_CODE,
      CAST(NULL AS STRING) AS RELIEVING_SF_STATUS_CODE,
      COALESCE(C.remarks, '') AS REASON,
      LOWER(RS.code) AS RELIEF_STATE,
      C.created_at AS RELIEF_CREATED_AT,
      -- Travel state from onsigner (reliever) assignment
      CASE SVA_ON.travel_documentation_status WHEN 'Completed' THEN 'travel_planning_done' WHEN 'Approved' THEN 'travel_planned' WHEN 'In-Progress' THEN 'travel_planning_started' WHEN 'InProgress' THEN 'travel_planning_started' WHEN 'Pending' THEN 'open' ELSE SVA_ON.travel_documentation_status END AS RELIEVER_TRAVEL_STATE,
      -- Documentation states from offsigner assignment (INT mapped to string)
      CASE SVA.flag_documentation_status WHEN 0 THEN 'open' WHEN 1 THEN 'documentation_started' WHEN 2 THEN 'documentation_completed' WHEN 3 THEN 'documentation_completed' ELSE NULL END AS FLAG_DOCUMENTATION_STATE,
      -- DOCUMENTATION_STATE: derived from SVA's granular doc status fields
      CASE 
        WHEN SVA.flag_documentation_status >= 2 
         AND SVA.general_documentation_status >= 2 
         AND SVA.joining_documentation_status >= 2
        THEN 'documentation_completed'
        ELSE 'open'
      END AS DOCUMENTATION_STATE,
      CASE SVA.general_documentation_status WHEN 0 THEN 'open' WHEN 1 THEN 'documentation_started' WHEN 2 THEN 'documentation_completed' WHEN 3 THEN 'documentation_completed' ELSE NULL END AS GENERAL_DOCUMENTATION_STATE,
      C.rank_id AS ONSIGNER_RANK_ID,
      C.joining_place_id AS SIGN_ON_PORT_ID,
      -- TRAVEL_REPLAN_STATE: not migrated per playbook (no SMAC equivalent)
      CAST(NULL AS STRING) AS TRAVEL_REPLAN_STATE,
      -- Travel state from offsigner (relieving) assignment
      CASE SVA.travel_documentation_status WHEN 'Completed' THEN 'travel_planning_done' WHEN 'Approved' THEN 'travel_planned' WHEN 'In-Progress' THEN 'travel_planning_started' WHEN 'InProgress' THEN 'travel_planning_started' WHEN 'Pending' THEN 'open' ELSE SVA.travel_documentation_status END AS RELIEVING_TRAVEL_STATE,
      -- Reliever (onsigner) details
      D.id AS RELIEVER_SEAFARER_ID,
      D.crew_code AS RELIEVER_CREW_CODE,
      COALESCE(D.first_name, Z.first_name) AS RELIEVER_FIRST_NAME,
      COALESCE(D.middle_name, Z.middle_name) AS RELIEVER_MIDDLE_NAME,
      COALESCE(D.last_name, Z.last_name) AS RELIEVER_LAST_NAME,
      -- Relieve (offsigner) details
      E.id AS RELIEVE_SEAFARER_ID,
      E.crew_code AS RELIEVE_CREW_CODE,
      E.first_name AS RELIEVE_FIRST_NAME,
      E.middle_name AS RELIEVE_MIDDLE_NAME,
      E.last_name AS RELIEVE_LAST_NAME,
      -- Relief Profile Link (SMAC URL)
      CONCAT('https://crewing.synergymarine.in/crewing/manning/relief/detail/', C.id) AS `Relief Profile Link`,
      -- CATEGORY_STATUS_BY_DAYS (same logic as SAC, using SVA contract dates)
      CASE
        WHEN DATEDIFF(DAY, CURRENT_DATE, CAST(SVA.contract_end_date AS DATE)) >= 0 THEN 'A. Crew onboard within contract'
        WHEN DATEDIFF(DAY, CURRENT_DATE, CAST(SVA.contract_end_date AS DATE)) BETWEEN -15 AND 0 THEN 'A. Crew onboard within contract'
        WHEN DATEDIFF(DAY, CURRENT_DATE, CAST(SVA.contract_end_date AS DATE)) BETWEEN -30 AND -15 THEN 'A. Crew onboard within contract'
        WHEN DATEDIFF(DAY, CURRENT_DATE, CAST(SVA.contract_end_date AS DATE)) BETWEEN -60 AND -30 THEN 'B. Delayed > 30 days'
        WHEN DATEDIFF(DAY, CURRENT_DATE, CAST(SVA.contract_end_date AS DATE)) BETWEEN -91 AND -60 THEN 'C. Delayed > 60 days'
        WHEN DATEDIFF(DAY, CURRENT_DATE, CAST(SVA.contract_end_date AS DATE)) BETWEEN -121 AND -91 THEN 'D. Delayed > 90 days'
        WHEN (DATEDIFF(MONTH, CAST(SVA.contract_start_date AS DATE), CAST(SVA.contract_end_date AS DATE)))
             + DATEDIFF(MONTH, CAST(SVA.contract_end_date AS DATE), CURRENT_DATE) BETWEEN 4 AND 10
        THEN 'G. Completed 4-10 Months'
        WHEN (DATEDIFF(MONTH, CAST(SVA.contract_start_date AS DATE), CAST(SVA.contract_end_date AS DATE)))
             + DATEDIFF(MONTH, CAST(SVA.contract_end_date AS DATE), CURRENT_DATE) = 10
        THEN 'G. Completed 4-10 Months'
        WHEN (DATEDIFF(MONTH, CAST(SVA.contract_start_date AS DATE), CAST(SVA.contract_end_date AS DATE)))
             + DATEDIFF(MONTH, CAST(SVA.contract_end_date AS DATE), CURRENT_DATE) = 11
        THEN 'H. Completed 11 Months'
        WHEN (DATEDIFF(MONTH, CAST(SVA.contract_start_date AS DATE), CAST(SVA.contract_end_date AS DATE)))
             + DATEDIFF(MONTH, CAST(SVA.contract_end_date AS DATE), CURRENT_DATE) >= 12
        THEN 'I. Completed > 12 Months'
      END AS CATEGORY_STATUS_BY_DAYS,
      CONCAT(DATEDIFF(MONTH, CAST(SVA.contract_start_date AS DATE), CAST(SVA.contract_end_date AS DATE)), ' Months') AS CONTRACT_TENURE,
      CONCAT(
        (DATEDIFF(MONTH, CAST(SVA.contract_start_date AS DATE), CAST(SVA.contract_end_date AS DATE)))
        + DATEDIFF(MONTH, CAST(SVA.contract_end_date AS DATE), CURRENT_DATE),
        ' Months'
      ) AS MONTHS,
      DATEADD(MONTH, 1, TO_DATE(SVA.contract_end_date)) AS `EXPIRY_DATE+1`,
      DATEADD(MONTH, -1, TO_DATE(SVA.contract_end_date)) AS `EXPIRY_DATE-1`,
      -- Rank Category (same CASE as SAC)
      CASE
        WHEN F.name IN ('Master', 'Chief Officer', 'Chief Engineer', 'Second Engineer') THEN 'Top 4 Rank'
        WHEN F.name IN (
          'Additional 3rd Officer','Additional Master','Additional Officer','Deck Cadet',
          'Deck Fitter','Junior Third Officer','Second Officer','Sr Deck cadet','Third Officer',
          'Trainee Master','DNS EXAM','Junior Watchkeeping Officer','Anchor Handler',
          'Additional Second Engineer','Assistant Engineer','Electrical Cadet','Electrical Officer',
          'Electrician','Electro Technical Officer','Engine Cadet','Fourth Engineer','Gas Engineer',
          'Junior Electrical Officer','Junior Engineer','Junior Fourth Engineer','Junior Gas Engineer',
          'SENIOR ELECTRICAL OFFICER','Third Engineer','Trainee Electrical Officer',
          'Trainee Gas Engineer','Trainee Marine Engineer','Trainee Radio Officer',
          'GME EXAM','Electrical Engineer','Deck Girl'
        ) THEN 'Officer'
        WHEN F.name IN (
          'Able Bodied Seaman','Bosun','Chief Cook','General Steward','Messboy','Messman',
          'Ordinary Seaman','Trainee General Steward','Trainee Ordinary seaman','Trainee Wiper',
          'Wiper','Wiper 1','Crew','Deck Fitter','Fitter','Motorman','Oiler','Oiler 1','Pumpman',
          'Trainee Pumpman','RPFW(Repair Fitter Welder)','Traine Fitter','Trainee Fitter',
          'Electro Technical Rating','Second Cook'
        ) THEN 'Rating'
        ELSE NULL
      END AS `Rank Category`,
      -- VESSEL_FLEET_TYPE (DRY/WET classification by vessel category)
      CASE
        WHEN UPPER(CAT_A.name) IN (
          'BULK CARRIER','CONTAINER','GEN CARGO / MULTI-PURPOSE VESSEL','CAR CARRIER / RO-RO',
          'CEMENT CARRIER','LOG CARRIER','REEFER CARGO','HEAVY LIFT/PROJECT CARGO',
          'WOODCHIP CARRIER','OBO CARRIER'
        ) THEN 'DRY'
        WHEN UPPER(CAT_A.name) IN (
          'OIL TANKER','CHEM/OIL PROD TANKER','ASPHALT / BITUMEN TANKER','LPG CARRIER (REFRI)',
          'CHEMICAL TANKER','LNG CARRIER','LPG CARRIER (PRESS)','SUPPLY /OFFSHORE / TUG BOAT / AHTS',
          'GAS TANKER','OIL/PROD BUNKER BARGE','CHEM/PROD TANKER'
        ) THEN 'WET'
        ELSE NULL
      END AS VESSEL_FLEET_TYPE,
      D.availability_date AS RELIVER_AVAILABILITY_DATE,
      Z.first_name AS SHORTLISTED_SEAFARER_FIRST_NAME,
      Z.middle_name AS SHORTLISTED_SEAFARER_MIDDLE_NAME,
      Z.last_name AS SHORTLISTED_SEAFARER_LAST_NAME,
      -- POD
      POD.POD AS POD_NAME,
      POD.vessel_name AS POD_VESSEL_NAME,
      -- Gender (SMAC uses gender_id FK instead of SAC '0'/'1' strings)
      COALESCE(GD.name, 'Unknown') AS RELIEVER_GENDER_NAME,
      GE.name AS RELIEVE_GENDER_NAME
    FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarer_reliefs C
      -- Offsigner's current sea experience on THIS vessel (sign_off_date IS NULL = currently onboard)
      INNER JOIN (
        SELECT * FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarer_sea_experiences
        WHERE deleted_at IS NULL AND sign_off_date IS NULL
      ) A ON A.seafarer_id = C.offsigner_id AND A.vessel_id = C.vessel_id
      -- Reliever's LATEST sea experience (only keep most recent per reliever)
      LEFT JOIN (
        SELECT * FROM (
          SELECT *, ROW_NUMBER() OVER (PARTITION BY seafarer_id ORDER BY sign_on_date DESC) AS rn
          FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarer_sea_experiences
          WHERE deleted_at IS NULL
        ) WHERE rn = 1
      ) L ON L.seafarer_id = C.onsigner_id
      -- Offsigner assignment (contract dates + doc statuses - replaces SAC VESSEL_CONTRACTS)
      LEFT JOIN curated_db.db_smac_prod_navitasai_crewing_public.seafarer_vessel_assignments SVA
        ON SVA.id = C.offsigner_assignment_id
      -- Onsigner assignment (reliever travel status)
      LEFT JOIN curated_db.db_smac_prod_navitasai_crewing_public.seafarer_vessel_assignments SVA_ON
        ON SVA_ON.id = C.onsigner_assignment_id
      -- Reliever (onsigner) seafarer
      LEFT JOIN (
        SELECT * FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarers
        WHERE deleted_at IS NULL
      ) D ON D.id = C.onsigner_id
      -- Relieving (offsigner) seafarer
      LEFT JOIN (
        SELECT * FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarers
        WHERE deleted_at IS NULL
      ) E ON E.id = C.offsigner_id
      -- Relief candidates (best confirmed candidate per relief: prefer Shortlisted=2 > Recommended=1)
      LEFT JOIN (
        SELECT * FROM (
          SELECT *, ROW_NUMBER() OVER (
            PARTITION BY relief_id
            ORDER BY state DESC, priority_order ASC
          ) AS rn
          FROM curated_db.db_smac_prod_navitasai_crewing_shore.relief_candidates
          WHERE deleted_at IS NULL
        ) WHERE rn = 1
      ) K ON K.relief_id = C.id
      -- Shortlisted seafarer details
      LEFT JOIN (
        SELECT * FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarers
        WHERE deleted_at IS NULL
      ) Z ON Z.id = K.seafarer_id
      -- Rank (from offsigner's current sea experience)
      LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.ranks F
        ON F.id = A.rank_id
      -- Nationality (of offsigner)
      LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.nationalities G
        ON G.id = E.nationality_id
      -- DOC/Contract Company (from sea experience doc_holder)
      LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.companies COMP
        ON COMP.id = A.doc_holder_company_id
      -- Vessel (for IMO number)
      LEFT JOIN curated_db.db_smac_prod_navitasai_masters_vessel.vessels V
        ON V.id = C.vessel_id
      -- Vessel category (from sea experience)
      LEFT JOIN curated_db.db_smac_prod_navitasai_masters_vessel.categories CAT_A
        ON CAT_A.id = A.vessel_category_id
      -- Relief state lookup
      LEFT JOIN curated_db.db_smac_prod_navitasai_masters_crewing.relief_states RS
        ON RS.id = C.relief_state_id
      -- POD (fleet lookup via vessel IMO)
      LEFT JOIN (
        SELECT DISTINCT f.name AS POD, v.imo_number, v.name AS vessel_name
        FROM curated_db.db_smac_prod_navitasai_masters_vessel.fleets f
          LEFT JOIN (
            SELECT * FROM curated_db.db_smac_prod_navitasai_masters_vessel.fld_fleet_vessels
            WHERE deleted_at IS NULL
          ) fv ON f.id = fv.fleet_id
          LEFT JOIN curated_db.db_smac_prod_navitasai_masters_vessel.vessels v ON v.id = fv.vessel_id
      ) POD ON POD.imo_number = V.imo_number
      -- Gender lookups (SMAC uses FK gender_id instead of SAC '0'/'1' strings)
      LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.genders GD
        ON GD.id = D.gender_id
      LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.genders GE
        ON GE.id = E.gender_id
    WHERE
      upper(coalesce(D.first_name, E.first_name)) NOT LIKE '%TEST%'
      AND upper(coalesce(D.last_name, E.last_name)) NOT LIKE '%TEST%'
      AND C.deleted_at IS NULL
      AND C.status = 1  -- Active relief record
      AND UPPER(RS.code) != 'CLOSE'  -- Exclude closed reliefs (equivalent of SAC B.STATUS = 'INFORCE')
  ),
  GEN_MONTH AS (
    SELECT DATEADD(MONTH, seq_num, TO_DATE('2000-01-01')) AS DELAYED_MONTH
    FROM (SELECT EXPLODE(SEQUENCE(0, 1199)) AS seq_num)
  )
  SELECT
    ST.*,
    MONTHNAME(GM.DELAYED_MONTH) AS DELAYED_MONTHS
  FROM SOURCE_TABLE ST
    LEFT JOIN GEN_MONTH GM
      ON GM.DELAYED_MONTH BETWEEN
        make_date(EXTRACT(YEAR FROM ST.`EXPIRY_DATE+1`), EXTRACT(MONTH FROM ST.`EXPIRY_DATE+1`), 1)
      AND
        make_date(
          EXTRACT(YEAR FROM COALESCE(ST.SIGN_OFF_DATE, CURRENT_DATE)),
          EXTRACT(MONTH FROM COALESCE(ST.SIGN_OFF_DATE, CURRENT_DATE)),
          1
        )
  ORDER BY CREW_CODE, VESSEL_IMO_NUMBER, SEAFARER_ID
) T
QUALIFY ROW_NUM = 1
ORDER BY CATEGORY_STATUS_BY_DAYS, DELAYED_MONTHS;

In [0]:
%sql
-- =============================================================================
-- SMAC PLANNER_VIEW - Migration from SAC
-- Source: reporting_layer.sac_prod_seafarer_public.PLANNER_VIEW
-- Target: reporting_layer.smac_prod.planner_view
--
-- STRUCTURE: revised_base_view (deduplicated 1 per seafarer)
--   LEFT JOIN revised_relief_view ON RELIEVE_SEAFARER_ID = REVISED_SEAFARER_ID
--   + 4 COMBINED columns (vessel name, category, CDC, bulk type)
-- =============================================================================

CREATE OR REPLACE TABLE reporting_layer.smac_prod.planner_view
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
)
AS
(
  SELECT
    * EXCEPT (ROW_NUM)
  FROM
    (
      SELECT
        A.*,
        B.* EXCEPT (ROW_NUM),
        COALESCE(B.VESSEL_NAME, A.REVISED_VESSEL_NAME) AS COMBINED_VESSEL_NAME,
        COALESCE(B.VESSEL_CATEGORY_NAME, A.REVISED_VESSEL_TYPE) AS COMBINED_VESSEL_CATEGORY_NAME,
        COALESCE(B.CDC_NUMBER, A.REVISED_CDC_NUMBER) AS COMBINED_CDC_NUMBER,
        CASE
          WHEN UPPER(TRIM(COALESCE(B.VESSEL_NAME, A.REVISED_VESSEL_NAME))) IN (
            'GENCO AQUITAINE','GENCO ARDENNES','GENCO AUVERGNE','GENCO BOURGOGNE','GENCO BRITTANY',
            'GENCO CONSTANTINE','GENCO DEFENDER','GENCO ENDEAVOUR','GENCO HUNTER','GENCO LANGUEDOC',
            'GENCO PICARDY','GENCO PREDATOR','GENCO PYRENEES','GENCO RESOLUTE','GENCO RHONE',
            'GENCO WARRIOR','GENCO TIBERIUS','GENCO TIGER','GENCO AUGUSTUS','GENCO TITUS',
            'GENCO COMMODUS','BALTIC WOLF','GENCO CLAUDIUS','GENCO HADRIAN','GENCO LONDON',
            'BALTIC MANTIS','BALTIC WASP','GENCO LION','BALTIC SCORPION','BALTIC HORNET',
            'GENCO MAXIMUS','BALTIC BEAR','GENCO ENTERPRISE','GENCO MADELEINE','GENCO MAYFLOWER',
            'GENCO CONSTELLATION','GENCO MARY','GENCO LADDEY','GENCO LIBERTY','GENCO COLUMBIA',
            'GENCO WEATHERLY','GENCO MAGIC','GENCO VIGILANT','GENCO FREEDOM','GENCO RANGER'
          ) THEN 'Bulk1'
          WHEN UPPER(TRIM(COALESCE(B.VESSEL_NAME, A.REVISED_VESSEL_NAME))) IN (
            'AMIS ACE','AMIS BRAVE','AMIS ELEGANCE','AMIS FORTUNE','AMIS INTEGRITY',
            'AMIS JUSTICE','AMIS KALON','AMIS LEADER','AMIS POWER','AMIS WISDOM I',
            'AMIS WISDOM II','AMIS WISDOM III','BLUE HORIZON','BUNUN KALON','BUNUN ORCHID',
            'BUNUN WISDOM','BUNUN XCEL','BUNUN YOUTH','BUNUN ZEST','CLEAR HORIZON',
            'COREFORTUNE OL','COREOCEAN OL','CORESKY OL','DAIWAN HERO','DAIWAN INFINITY',
            'DAIWAN KALON','DAIWAN WISDOM','ETERNITY SW','FRONTIER BONANZA','GLOBAL FAITH',
            'GOLDEN KIKU','KATAGALAN ACE','KATAGALAN BRAVE','MOONBRIGHT SW','NALUHU',
            'POAVOSA WISDOM','POAVOSA WISDOM III','SAKIZAYA ACE','SAKIZAYA CHAMPION',
            'SAKIZAYA RESPECT','SAKIZAYA XCEL','SCARLET EAGLE','TAOKAS WISDOM','TRANSFORMER OL'
          ) THEN 'Bulk2'
          WHEN UPPER(TRIM(COALESCE(B.VESSEL_NAME, A.REVISED_VESSEL_NAME))) IN (
            'OCEAN GLSR','IBERIAN BULKER','AFRICAN BULKER','BERGE NYANGANI','WORLD RUBY',
            'WORLD PRIZE','WORLD DIANA','WORLD VIRTUE','NORD ENERGY','NORD POWER','MESK',
            'ASIAN BULKER','AUSTRALIAN BULKER','ICELAND BULKER','NORD MAGNES','LAVENDER',
            'LOWLANDS BLUE','LOWLANDS DAWN','LOWLANDS FUTURE','LOWLANDS HORIZON','LOWLANDS RISE',
            'LOWLANDS SAGE','LOWLANDS OPAL','LOWLANDS YELLOW','ACE ETERNITY','AN LI','BENITAMOU',
            'GLOBAL HARMONY','MH ADAGIO','NEO','OCCITAN KEY','OCCITAN PAUILLAC','OCCITAN SKY',
            'SATIGNY','KEY JOURNEY','MH ARPEGGIO','NORD FERRUM','K. IRON MOUNTAIN','K RUBY',
            'K VICTORY','K PREMIUM ORE','K CONFIDENCE','IRON PHOENIX','IRON MIRACLE',
            'NORD STEEL','SERVETTE','MIRACLE'
          ) THEN 'Bulk3'
          WHEN UPPER(TRIM(COALESCE(B.VESSEL_NAME, A.REVISED_VESSEL_NAME))) IN (
            'CARL OLDENDORFF','CHARLOTTE OLDENDORFF','CHRISTINE OLDENDORFF','CONRAD OLDENDORFF',
            'CORA OLDENDORFF','KIM OLDENDORFF','KLARA OLDENDORFF','KNUT OLDENDORFF','EGE-M',
            'PATRICIA OLDENDORFF','PAUL OLDENDORFF','PENELOPE OLDENDORFF','PETER OLDENDORFF',
            'PHILIPP OLDENDORFF','PIA OLDENDORFF','EPICURUS','ETOILE','TRUE NEPTUNE',
            'TRUE CONRAD','TRUE CHAMPION','TAURUS','CL XUCHANG','CL CHANGSHA','CL LUZHOU',
            'CL YANGZHOU','CL GANJIANG','CL LIANYUNGANG','PACIFIC EAST','PACIFIC WEST',
            'PACIFIC NORTH','PACIFIC SOUTH','KM HAKATA','AM OCEAN SILVER','AM OCEAN STAR',
            'VINAYAK','LAMPARD','ZOLA','GH KAHLO','AM UMANG','AM KIRTI','AM TARANG',
            'GCL YAMUNA','GCL SABARMATI','GCL GANGA','GCL TAPI','GCL NARMADA','GCL MAHANADI'
          ) THEN 'Bulk4'
          ELSE 'Others'
        END AS COMBINED_BULK_TYPE,
        ROW_NUMBER() OVER (
            PARTITION BY A.REVISED_SEAFARER_ID
            ORDER BY
              A.REVISED_LATEST_DATE,
              A.REVISED_LATEST_CONTRACT_END_DATE,
              A.REVISED_LATEST_SIGN_OFF_DATE DESC
          ) AS ROW_NUM
      FROM
        (
          SELECT
            *
          FROM
            (
              SELECT
                T.*,
                ROW_NUMBER() OVER (
                    PARTITION BY REVISED_SEAFARER_ID
                    ORDER BY
                      REVISED_LATEST_DATE,
                      REVISED_LATEST_CONTRACT_END_DATE,
                      REVISED_LATEST_SIGN_OFF_DATE DESC
                  ) AS ROW_NUM1
              FROM
                (
                  SELECT
                    A.FIRST_NAME AS REVISED_FIRST_NAME,
                    A.MIDDLE_NAME AS REVISED_MIDDLE_NAME,
                    A.LAST_NAME AS REVISED_LAST_NAME,
                    A.DATE AS REVISED_LATEST_DATE,
                    A.CREW_CODE AS REVISED_CREW_CODE,
                    A.VESSEL_CATEGORY_NAME AS REVISED_VESSEL_TYPE,
                    A.VESSEL_NAME AS REVISED_VESSEL_NAME,
                    A.SEAFARER_ID AS REVISED_SEAFARER_ID,
                    A.CURRENT_STATUS AS REVISED_current_status,
                    A.CURRENT_RANK_NAME AS REVISED_RANK_NAME,
                    A.SIGN_OFF_REASON AS REVISED_SIGN_OFF_REASON,
                    A.AVAILABILITY_DATE AS REVISED_AVAILABILITY_DATE,
                    A.SHIP_MANAGEMENT_COMPANY_NAME AS REVISED_SHIP_MANAGEMENT_COMPANY_NAME,
                    A.GENDER_NAME AS REVISED_GENDER_NAME,
                    A.NATIONALITY_NAME AS REVISED_NATIONALITY_NAME,
                    A.SIGN_ON_DATE AS REVISED_SIGN_ON_DATE,
                    A.SIGN_OFF_DATE AS REVISED_SIGN_OFF_DATE,
                    A.CONTRACT_END_DATE AS REVISED_CONTRACT_END_DATE,
                    A.LATEST_CONTRACT_END_DATE AS REVISED_LATEST_CONTRACT_END_DATE,
                    A.LATEST_SIGN_OFF_DATE AS REVISED_LATEST_SIGN_OFF_DATE,
                    A.`Overdue by / Days left` AS `Revised Overdue by / Days left`,
                    A.PROFILE_STATUS AS REVISED_PROFILE_STATUS,
                    A.CDC_NUMBER AS REVISED_CDC_NUMBER,
                    -- REVISED_BULK_TYPE (same vessel name classification)
                    CASE
                      WHEN UPPER(TRIM(A.VESSEL_NAME)) IN (
                        'GENCO AQUITAINE','GENCO ARDENNES','GENCO AUVERGNE','GENCO BOURGOGNE','GENCO BRITTANY',
                        'GENCO CONSTANTINE','GENCO DEFENDER','GENCO ENDEAVOUR','GENCO HUNTER','GENCO LANGUEDOC',
                        'GENCO PICARDY','GENCO PREDATOR','GENCO PYRENEES','GENCO RESOLUTE','GENCO RHONE',
                        'GENCO WARRIOR','GENCO TIBERIUS','GENCO TIGER','GENCO AUGUSTUS','GENCO TITUS',
                        'GENCO COMMODUS','BALTIC WOLF','GENCO CLAUDIUS','GENCO HADRIAN','GENCO LONDON',
                        'BALTIC MANTIS','BALTIC WASP','GENCO LION','BALTIC SCORPION','BALTIC HORNET',
                        'GENCO MAXIMUS','BALTIC BEAR','GENCO ENTERPRISE','GENCO MADELEINE','GENCO MAYFLOWER',
                        'GENCO CONSTELLATION','GENCO MARY','GENCO LADDEY','GENCO LIBERTY','GENCO COLUMBIA',
                        'GENCO WEATHERLY','GENCO MAGIC','GENCO VIGILANT','GENCO FREEDOM','GENCO RANGER'
                      ) THEN 'Bulk1'
                      WHEN UPPER(TRIM(A.VESSEL_NAME)) IN (
                        'AMIS ACE','AMIS BRAVE','AMIS ELEGANCE','AMIS FORTUNE','AMIS INTEGRITY',
                        'AMIS JUSTICE','AMIS KALON','AMIS LEADER','AMIS POWER','AMIS WISDOM I',
                        'AMIS WISDOM II','AMIS WISDOM III','BLUE HORIZON','BUNUN KALON','BUNUN ORCHID',
                        'BUNUN WISDOM','BUNUN XCEL','BUNUN YOUTH','BUNUN ZEST','CLEAR HORIZON',
                        'COREFORTUNE OL','COREOCEAN OL','CORESKY OL','DAIWAN HERO','DAIWAN INFINITY',
                        'DAIWAN KALON','DAIWAN WISDOM','ETERNITY SW','FRONTIER BONANZA','GLOBAL FAITH',
                        'GOLDEN KIKU','KATAGALAN ACE','KATAGALAN BRAVE','MOONBRIGHT SW','NALUHU',
                        'POAVOSA WISDOM','POAVOSA WISDOM III','SAKIZAYA ACE','SAKIZAYA CHAMPION',
                        'SAKIZAYA RESPECT','SAKIZAYA XCEL','SCARLET EAGLE','TAOKAS WISDOM','TRANSFORMER OL'
                      ) THEN 'Bulk2'
                      WHEN UPPER(TRIM(A.VESSEL_NAME)) IN (
                        'OCEAN GLSR','IBERIAN BULKER','AFRICAN BULKER','BERGE NYANGANI','WORLD RUBY',
                        'WORLD PRIZE','WORLD DIANA','WORLD VIRTUE','NORD ENERGY','NORD POWER','MESK',
                        'ASIAN BULKER','AUSTRALIAN BULKER','ICELAND BULKER','NORD MAGNES','LAVENDER',
                        'LOWLANDS BLUE','LOWLANDS DAWN','LOWLANDS FUTURE','LOWLANDS HORIZON','LOWLANDS RISE',
                        'LOWLANDS SAGE','LOWLANDS OPAL','LOWLANDS YELLOW','ACE ETERNITY','AN LI','BENITAMOU',
                        'GLOBAL HARMONY','MH ADAGIO','NEO','OCCITAN KEY','OCCITAN PAUILLAC','OCCITAN SKY',
                        'SATIGNY','KEY JOURNEY','MH ARPEGGIO','NORD FERRUM','K. IRON MOUNTAIN','K RUBY',
                        'K VICTORY','K PREMIUM ORE','K CONFIDENCE','IRON PHOENIX','IRON MIRACLE',
                        'NORD STEEL','SERVETTE','MIRACLE'
                      ) THEN 'Bulk3'
                      WHEN UPPER(TRIM(A.VESSEL_NAME)) IN (
                        'CARL OLDENDORFF','CHARLOTTE OLDENDORFF','CHRISTINE OLDENDORFF','CONRAD OLDENDORFF',
                        'CORA OLDENDORFF','KIM OLDENDORFF','KLARA OLDENDORFF','KNUT OLDENDORFF','EGE-M',
                        'PATRICIA OLDENDORFF','PAUL OLDENDORFF','PENELOPE OLDENDORFF','PETER OLDENDORFF',
                        'PHILIPP OLDENDORFF','PIA OLDENDORFF','EPICURUS','ETOILE','TRUE NEPTUNE',
                        'TRUE CONRAD','TRUE CHAMPION','TAURUS','CL XUCHANG','CL CHANGSHA','CL LUZHOU',
                        'CL YANGZHOU','CL GANJIANG','CL LIANYUNGANG','PACIFIC EAST','PACIFIC WEST',
                        'PACIFIC NORTH','PACIFIC SOUTH','KM HAKATA','AM OCEAN SILVER','AM OCEAN STAR',
                        'VINAYAK','LAMPARD','ZOLA','GH KAHLO','AM UMANG','AM KIRTI','AM TARANG',
                        'GCL YAMUNA','GCL SABARMATI','GCL GANGA','GCL TAPI','GCL NARMADA','GCL MAHANADI'
                      ) THEN 'Bulk4'
                      ELSE 'Others'
                    END AS REVISED_BULK_TYPE,
                    -- Revised Rank Category
                    CASE
                      WHEN A.CURRENT_RANK_NAME IN ('Master','Chief Officer','Chief Engineer','Second Engineer')
                      THEN 'Top 4 Rank'
                      WHEN A.CURRENT_RANK_NAME IN (
                        'Additional 3rd Officer','Additional Master','Additional Officer','Deck Cadet',
                        'Deck Fitter','Junior Third Officer','Second Officer','Sr Deck cadet','Third Officer',
                        'Trainee Master','DNS EXAM','Junior Watchkeeping Officer','Anchor Handler',
                        'Additional Second Engineer','Assistant Engineer','Electrical Cadet','Electrical Officer',
                        'Electrician','Electro Technical Officer','Engine Cadet','Fourth Engineer','Gas Engineer',
                        'Junior Electrical Officer','Junior Engineer','Junior Fourth Engineer','Junior Gas Engineer',
                        'SENIOR ELECTRICAL OFFICER','Third Engineer','Trainee Electrical Officer',
                        'Trainee Gas Engineer','Trainee Marine Engineer','Trainee Radio Officer',
                        'GME EXAM','Electrical Engineer','Deck Girl'
                      ) THEN 'Officer'
                      WHEN A.CURRENT_RANK_NAME IN (
                        'Able Bodied Seaman','Bosun','Chief Cook','General Steward','Messboy','Messman',
                        'Ordinary Seaman','Trainee General Steward','Trainee Ordinary seaman','Trainee Wiper',
                        'Wiper','Wiper 1','Crew','Deck Fitter','Fitter','Motorman','Oiler','Oiler 1','Pumpman',
                        'Trainee Pumpman','RPFW(Repair Fitter Welder)','Traine Fitter','Trainee Fitter',
                        'Electro Technical Rating','Second Cook'
                      ) THEN 'Rating'
                      ELSE NULL
                    END AS `Revised Rank Category`
                  FROM
                    reporting_layer.smac_prod.revised_base_view A
                ) T
            )
          WHERE
            ROW_NUM1 = 1
        ) A
          LEFT JOIN reporting_layer.smac_prod.revised_relief_view B
            ON B.RELIEVE_SEAFARER_ID = A.REVISED_SEAFARER_ID
    )
  WHERE
    ROW_NUM1 = 1
)

## Section 3: Appraisal Views

`add_digital_appraisal_view` → `appraisal_performance` → `digital_appraisal_view`

Run in order: `digital_appraisal_view` depends on `appraisal_performance`.

In [0]:
%sql
-- =============================================================================
-- SMAC add_digital_appraisal_view - Migration from SAC
-- Source: reporting_layer.sac_prod_seafarer_public.add_digital_appraisal_view
-- Target: reporting_layer.smac_prod.add_digital_appraisal_view
--
-- MAPPING:
--   SAC appraisals (FEEDBACK JSON explode) → SMAC seafarer_appraisal_forms (1 row per form)
--   SAC feedback.templateName → SMAC appraisal_stages.name (via stage_id)
--   SAC appraisals.CREATED_BY_NAME → SMAC seafarer_appraisals.initiated_by → idp_public.user_profiles
--   SAC feedback.appraiser_name → SMAC seafarer_appraisal_forms.assigned_to_user_id, resolved by stage_type:
--       stage_type = 'Appraisee'         -> parent appraisal's seafarer (ap.seafarer_id) + rank suffix
--                                            (assigned_to_user_id is an empty GUID for this stage by design/migration
--                                             - never join on it for Appraisee)
--       assigned_to_user_type = 'Shore'  -> idp_public.user_profiles
--       else (assigned_to_user_type = 'Seafarer') -> crewing_public.seafarers
--     audit_info.notes regex kept as a fallback for the ~20% of forms where the FK is unresolved
--     (2026-07-29 update: replaced audit_info.notes regex as the primary source for Intiated_by /
--      appraiser names - see SMAC Migration/Validation Docs for the validation that drove this change)
--   SAC RANK_ID IN ('13','18') → SMAC ranks.name IN ('Master','Chief Engineer')
-- =============================================================================

CREATE OR REPLACE TABLE reporting_layer.smac_prod.add_digital_appraisal_view
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
)
AS
SELECT *
FROM (
  SELECT
    f.form_status AS `feedback Status`,
    ap.appraisal_status AS `Appraisal Status`,
    at_tbl.name AS `AppraisalType`,
    COALESCE(stg.name, f.stage_type) AS `feedback_templatename`,
    ap.seafarer_id AS SEAFARER_ID,
    -- Initiated By: seafarer_appraisals.initiated_by -> IDP user_profiles
    -- (fallback to audit_info notes regex when initiated_by FK is unresolved)
    COALESCE(
      NULLIF(TRIM(CONCAT_WS(' ', IB.first_name, IB.last_name)), ''),
      NULLIF(
        REGEXP_EXTRACT(get_json_object(ap.audit_info, '$.notes'), 'Created by:\\s*(?:AHOY-)?([^;-]+)', 1),
        ''
      )
    ) AS `Intiated_by`,
    SF.crew_code AS CREW_CODE,
    CONCAT(
      COALESCE(SF.first_name, ''), ' ',
      COALESCE(SF.middle_name, ''), ' ',
      COALESCE(SF.last_name, '')
    ) AS SEAFARER_NAME,
    CONCAT(
      'https://crewing.synergymarine.in/crewing/seafarer/details/',
      SF.id, '/appraisals'
    ) AS APPRAISAL_LINK,
    R.name AS CURRENT_RANK_NAME,
    ap.period_from AS FROM_DATE,
    ap.period_to AS TO_DATE,
    V.name AS VESSEL_NAME,
    CAT.name AS VESSEL_TYPE,
    -- Feedback comments: prefer the appraiser's final sign-off remark in confirmation_data.Remarks
    -- (2026-07-29 fix: confirmation_data.Remarks is the actual comment surfaced in SAC's report for
    --  this field - confirmed by an exact-text match against a known-good SAC value. Falls back to
    --  the $.data section regex for stages without confirmation_data, e.g. Appraisee Acknowledgement.
    --  ~43% combined coverage vs ~14% from the $.data regex alone.)
    COALESCE(
      NULLIF(get_json_object(f.confirmation_data, '$.Remarks'), ''),
      NULLIF(
        array_join(
          regexp_extract_all(
            COALESCE(get_json_object(f.submission_data, '$.data'), ''),
            '"[^"]*[Rr]emarks?"\\s*:\\s*"([^"]+)"',
            1
          ),
          '; '
        ),
        ''
      )
    ) AS FEEDBACK_COMMENTS,
    -- Appraiser / reviewer name, resolved by stage_type / assigned_to_user_type (see mapping note above);
    -- falls back to the old audit_info notes regex when the assignee FK is unresolved
    COALESCE(
      NULLIF(
        CASE
          WHEN UPPER(TRIM(f.stage_type)) = 'APPRAISEE' THEN
            CONCAT(
              TRIM(CONCAT_WS(' ', SF.first_name, SF.middle_name, SF.last_name)),
              CASE WHEN R.name IS NOT NULL THEN CONCAT(' (', R.name, ')') ELSE '' END
            )
          WHEN UPPER(TRIM(f.assigned_to_user_type)) = 'SHORE' THEN
            TRIM(CONCAT_WS(' ', UP_ASG.first_name, UP_ASG.last_name))
          ELSE
            TRIM(CONCAT_WS(' ', S_ASG.first_name, S_ASG.middle_name, S_ASG.last_name))
        END,
        ''
      ),
      NULLIF(
        REGEXP_REPLACE(COALESCE(get_json_object(f.audit_info, '$.notes'), ''), '^Appraiser:\\s*', ''),
        ''
      )
    ) AS `feedback Appraiser Name`,
    stg.name AS `feedback templateName`
  FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarer_appraisals ap
    -- Appraisal forms (one per feedback stage - equivalent of SAC FEEDBACK JSON explode)
    INNER JOIN curated_db.db_smac_prod_navitasai_crewing_public.seafarer_appraisal_forms f
      ON f.appraisal_id = ap.id AND f.deleted_at IS NULL
    -- Appraisal stages (templateName equivalent)
    LEFT JOIN curated_db.db_smac_prod_navitasai_masters_crewing.appraisal_stages stg
      ON stg.id = f.stage_id
    -- Seafarer details (also used to resolve the Appraisee stage's own name)
    LEFT JOIN (
      SELECT * FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarers
      WHERE deleted_at IS NULL
    ) SF ON SF.id = ap.seafarer_id
    -- Appraisal type
    INNER JOIN curated_db.db_smac_prod_navitasai_masters_crewing.appraisal_types at_tbl
      ON at_tbl.id = ap.appraisal_type_id
    -- Rank (also used for the Appraisee stage's "(Master)" style suffix)
    LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.ranks R
      ON R.id = ap.rank_id
    -- Vessel
    LEFT JOIN curated_db.db_smac_prod_navitasai_masters_vessel.vessels V
      ON V.id = ap.vessel_id
    -- Vessel category
    LEFT JOIN curated_db.db_smac_prod_navitasai_masters_vessel.categories CAT
      ON CAT.id = ap.vessel_type_id
    -- Initiated By resolution
    LEFT JOIN curated_db.db_smac_prod_navitasai_idp_public.user_profiles IB
      ON IB.id = ap.initiated_by
    -- Shore assignee resolution (Appraisee stage is resolved from SF/R above instead)
    LEFT JOIN curated_db.db_smac_prod_navitasai_idp_public.user_profiles UP_ASG
      ON UP_ASG.id = f.assigned_to_user_id
     AND UPPER(TRIM(f.assigned_to_user_type)) = 'SHORE'
     AND UPPER(TRIM(f.stage_type)) <> 'APPRAISEE'
    -- Seafarer assignee resolution (e.g. an onboard reviewer)
    LEFT JOIN curated_db.db_smac_prod_navitasai_crewing_public.seafarers S_ASG
      ON S_ASG.id = f.assigned_to_user_id
     AND UPPER(TRIM(f.assigned_to_user_type)) = 'SEAFARER'
     AND UPPER(TRIM(f.stage_type)) <> 'APPRAISEE'
  WHERE
    ap.deleted_at IS NULL
    AND R.name IN ('Master', 'Chief Engineer')
)
  PIVOT (
    MAX(`feedback Appraiser Name`) FOR `feedback templateName` IN (
      'Crewing Superintendent Feedback',
      'Marine Superintendent Feedback',
      'Technical Superintendent Feedback',
      'Appraisee Acknowledgement' AS `Appraisee feedback`,
      'Marine Manager Feedback',
      'Technical Manager Feedback'
    )
  )
ORDER BY SEAFARER_ID, FROM_DATE DESC

In [0]:
%sql
-- =============================================================================
-- SMAC appraisal_performance - Migration from SAC
-- Source: reporting_layer.sac_prod_seafarer_public.appraisal_performance
-- Target: reporting_layer.smac_prod.appraisal_performance
--
-- SAC LOGIC:
--   1. Gets closed appraisals, explodes FEEDBACK JSON array
--   2. Extracts $.rating per feedback template
--   3. Averages rating across all templates per appraisal
--   4. Groups by (seafarer, created_at, from_date, to_date)
--
-- SMAC MAPPING:
--   SAC appraisals.FEEDBACK[].rating → SMAC seafarer_appraisal_forms.average_score
--   SAC appraisals.status = 'Closed' → SMAC appraisal_status = 'closed'
--   SAC seafarer_id (INT) → SMAC seafarer_id (UUID)
--   SAC from_date/to_date → SMAC period_from/period_to
-- =============================================================================

CREATE OR REPLACE TABLE reporting_layer.smac_prod.appraisal_performance
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
)
AS
SELECT
  ap.seafarer_id,
  ap.created_at,
  sf.crew_code,
  ap.period_to AS to_date,
  ap.period_from AS from_date,
  ROUND(AVG(f.average_score), 2) AS performance_rating
FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarer_appraisals ap
INNER JOIN curated_db.db_smac_prod_navitasai_crewing_public.seafarer_appraisal_forms f
  ON f.appraisal_id = ap.id AND f.deleted_at IS NULL AND f.average_score > 0
LEFT JOIN (
  SELECT id, crew_code FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarers
  WHERE deleted_at IS NULL
) sf ON sf.id = ap.seafarer_id
WHERE ap.deleted_at IS NULL
  AND ap.appraisal_status = 'closed'
GROUP BY ap.seafarer_id, ap.created_at, sf.crew_code, ap.period_to, ap.period_from
HAVING performance_rating <> 0
ORDER BY sf.crew_code, ap.period_from DESC;

In [0]:
%sql
-- =============================================================================
-- SMAC digital_appraisal_view - Migration from SAC
-- Source: reporting_layer.sac_prod_seafarer_public.digital_appraisal_view
-- Target: reporting_layer.smac_prod.digital_appraisal_view
--
-- LOGIC: Latest appraisal per seafarer (RNUM=1) with per-stage:
--   - Appraiser Name, Form Status, Feedback Comments
--   + performance_rating from appraisal_performance table
--
-- 2026-07-29 fix: Feedback Comment columns now prefer f.confirmation_data.$.Remarks
--   (the appraiser's final sign-off remark, ~43% coverage - same fix already applied to
--   add_digital_appraisal_view) and fall back to the old submission_data.$.data regex
--   (~14% coverage alone) for stages without confirmation_data (e.g. Appraisee Acknowledgement).
-- =============================================================================

CREATE OR REPLACE TABLE reporting_layer.smac_prod.digital_appraisal_view
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
)
AS
SELECT * FROM (
  SELECT
    T1.*,
    PR.performance_rating,
    ROW_NUMBER() OVER (PARTITION BY T1.SEAFARER_ID ORDER BY T1.FROM_DATE DESC) AS RNUM
  FROM (
    SELECT
      ap.appraisal_status AS `Appraisal Status`,
      at_tbl.name AS `AppraisalType`,
      ap.seafarer_id AS SEAFARER_ID,
      REGEXP_EXTRACT(
        get_json_object(ap.audit_info, '$.notes'),
        'Created by:\\s*(?:AHOY-)?([^;-]+)', 1
      ) AS `Intiated_by`,
      SF.crew_code AS CREW_CODE,
      CONCAT(COALESCE(SF.first_name, ''), ' ', COALESCE(SF.middle_name, ''), ' ', COALESCE(SF.last_name, '')) AS SEAFARER_NAME,
      CONCAT('https://crewing.synergymarine.in/crewing/seafarer/details/', SF.id, '/appraisals') AS APPRAISAL_LINK,
      R.name AS CURRENT_RANK_NAME,
      ap.period_from AS FROM_DATE,
      ap.period_to AS TO_DATE,
      V.name AS VESSEL_NAME,
      CAT.name AS VESSEL_TYPE,
      -- Crewing Superintendent
      MAX(CASE WHEN stg.name = 'Crewing Superintendent Feedback' THEN REGEXP_REPLACE(COALESCE(get_json_object(f.audit_info, '$.notes'), ''), '^Appraiser:\\s*', '') END) AS `Crewing Superintendent Feedback`,
      MAX(CASE WHEN stg.name = 'Crewing Superintendent Feedback' THEN f.form_status END) AS `Crewing Superintendent Feedback Status`,
      MAX(CASE WHEN stg.name = 'Crewing Superintendent Feedback' THEN COALESCE(NULLIF(get_json_object(f.confirmation_data, '$.Remarks'), ''), NULLIF(array_join(regexp_extract_all(COALESCE(get_json_object(f.submission_data, '$.data'), ''), '"[^"]*[Rr]emarks?"\\s*:\\s*"([^"]+)"', 1), '; '), '')) END) AS `Crewing Superintendent Feedback Comment`,
      -- Marine Superintendent
      MAX(CASE WHEN stg.name = 'Marine Superintendent Feedback' THEN REGEXP_REPLACE(COALESCE(get_json_object(f.audit_info, '$.notes'), ''), '^Appraiser:\\s*', '') END) AS `Marine Superintendent Feedback`,
      MAX(CASE WHEN stg.name = 'Marine Superintendent Feedback' THEN f.form_status END) AS `Marine Superintendent Feedback Status`,
      MAX(CASE WHEN stg.name = 'Marine Superintendent Feedback' THEN COALESCE(NULLIF(get_json_object(f.confirmation_data, '$.Remarks'), ''), NULLIF(array_join(regexp_extract_all(COALESCE(get_json_object(f.submission_data, '$.data'), ''), '"[^"]*[Rr]emarks?"\\s*:\\s*"([^"]+)"', 1), '; '), '')) END) AS `Marine Superintendent Feedback Comment`,
      -- Technical Superintendent
      MAX(CASE WHEN stg.name = 'Technical Superintendent Feedback' THEN REGEXP_REPLACE(COALESCE(get_json_object(f.audit_info, '$.notes'), ''), '^Appraiser:\\s*', '') END) AS `Technical Superintendent Feedback`,
      MAX(CASE WHEN stg.name = 'Technical Superintendent Feedback' THEN f.form_status END) AS `Technical Superintendent Feedback Status`,
      MAX(CASE WHEN stg.name = 'Technical Superintendent Feedback' THEN COALESCE(NULLIF(get_json_object(f.confirmation_data, '$.Remarks'), ''), NULLIF(array_join(regexp_extract_all(COALESCE(get_json_object(f.submission_data, '$.data'), ''), '"[^"]*[Rr]emarks?"\\s*:\\s*"([^"]+)"', 1), '; '), '')) END) AS `Technical Superintendent Feedback Comment`,
      -- Appraisee
      MAX(CASE WHEN stg.name = 'Appraisee Acknowledgement' THEN REGEXP_REPLACE(COALESCE(get_json_object(f.audit_info, '$.notes'), ''), '^Appraiser:\\s*', '') END) AS `Appraisee feedback`,
      MAX(CASE WHEN stg.name = 'Appraisee Acknowledgement' THEN f.form_status END) AS `Appraisee feedback Status`,
      MAX(CASE WHEN stg.name = 'Appraisee Acknowledgement' THEN COALESCE(NULLIF(get_json_object(f.confirmation_data, '$.Remarks'), ''), NULLIF(array_join(regexp_extract_all(COALESCE(get_json_object(f.submission_data, '$.data'), ''), '"[^"]*[Rr]emarks?"\\s*:\\s*"([^"]+)"', 1), '; '), '')) END) AS `Appraisee feedback Comment`,
      -- Marine Manager
      MAX(CASE WHEN stg.name = 'Marine Manager Feedback' THEN REGEXP_REPLACE(COALESCE(get_json_object(f.audit_info, '$.notes'), ''), '^Appraiser:\\s*', '') END) AS `Marine Manager Feedback`,
      MAX(CASE WHEN stg.name = 'Marine Manager Feedback' THEN f.form_status END) AS `Marine Manager Feedback Status`,
      MAX(CASE WHEN stg.name = 'Marine Manager Feedback' THEN COALESCE(NULLIF(get_json_object(f.confirmation_data, '$.Remarks'), ''), NULLIF(array_join(regexp_extract_all(COALESCE(get_json_object(f.submission_data, '$.data'), ''), '"[^"]*[Rr]emarks?"\\s*:\\s*"([^"]+)"', 1), '; '), '')) END) AS `Marine Manager Feedback Comment`,
      -- Technical Manager
      MAX(CASE WHEN stg.name = 'Technical Manager Feedback' THEN REGEXP_REPLACE(COALESCE(get_json_object(f.audit_info, '$.notes'), ''), '^Appraiser:\\s*', '') END) AS `Technical Manager Feedback`,
      MAX(CASE WHEN stg.name = 'Technical Manager Feedback' THEN f.form_status END) AS `Technical Manager Feedback Status`,
      MAX(CASE WHEN stg.name = 'Technical Manager Feedback' THEN COALESCE(NULLIF(get_json_object(f.confirmation_data, '$.Remarks'), ''), NULLIF(array_join(regexp_extract_all(COALESCE(get_json_object(f.submission_data, '$.data'), ''), '"[^"]*[Rr]emarks?"\\s*:\\s*"([^"]+)"', 1), '; '), '')) END) AS `Technical Manager Feedback Comment`
    FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarer_appraisals ap
    INNER JOIN curated_db.db_smac_prod_navitasai_crewing_public.seafarer_appraisal_forms f
      ON f.appraisal_id = ap.id AND f.deleted_at IS NULL
    LEFT JOIN curated_db.db_smac_prod_navitasai_masters_crewing.appraisal_stages stg ON stg.id = f.stage_id
    LEFT JOIN (SELECT * FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarers WHERE deleted_at IS NULL) SF ON SF.id = ap.seafarer_id
    INNER JOIN curated_db.db_smac_prod_navitasai_masters_crewing.appraisal_types at_tbl ON at_tbl.id = ap.appraisal_type_id
    LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.ranks R ON R.id = ap.rank_id
    LEFT JOIN curated_db.db_smac_prod_navitasai_masters_vessel.vessels V ON V.id = ap.vessel_id
    LEFT JOIN curated_db.db_smac_prod_navitasai_masters_vessel.categories CAT ON CAT.id = ap.vessel_type_id
    WHERE ap.deleted_at IS NULL AND R.name IN ('Master', 'Chief Engineer')
    GROUP BY ap.appraisal_status, at_tbl.name, ap.seafarer_id, ap.audit_info, SF.crew_code,
      SF.first_name, SF.middle_name, SF.last_name, SF.id, R.name, ap.period_from, ap.period_to,
      V.name, CAT.name
  ) T1
  LEFT JOIN reporting_layer.smac_prod.appraisal_performance PR
    ON PR.crew_code = T1.CREW_CODE AND PR.from_date = T1.FROM_DATE AND PR.to_date = T1.TO_DATE
)
WHERE RNUM = 1
ORDER BY CREW_CODE, FROM_DATE DESC;

## Section 4: Other Views

In [0]:
%sql
-- =============================================================================
-- SMAC inactive_seafarers - Migration from SAC
-- Source: reporting_layer.sac_prod_seafarer_public.inactive_seafarers
-- Target: reporting_layer.smac_prod.inactive_seafarers
--
-- SAC LOGIC:
--   1. Latest INACTIVE remark per seafarer from SEAFARER_REMARKS
--   2. Join to SEAFARER_PROFILE_REMARKS for reason name
--   3. Join to latest SEA_EXPERIENCE for sign-off/vessel info
--   4. Classify INACTIVE_REASON (BT/UT/Joined elsewhere)
--
-- SMAC MAPPING:
--   SAC SEAFARER_REMARKS → SMAC seafarer_remarks (crewing_shore)
--   SAC remark_type='INACTIVE' → SMAC profile_remark_type_id = DEACTIVATION
--   SAC SEAFARER_PROFILE_REMARKS → SMAC profile_remark_reasons
-- =============================================================================

CREATE OR REPLACE TABLE reporting_layer.smac_prod.inactive_seafarers
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
)
AS
SELECT DISTINCT
  SR.id AS ID,
  S.id AS SEAFARER_ID,
  SR.profile_remark_reason_id AS REMARK_IDENTIFIER,
  S.crew_code AS CREW_CODE,
  CONCAT(COALESCE(S.first_name, ''), ' ', COALESCE(S.last_name, '')) AS CREW_NAME,
  PRR.description AS NAME,
  N.name AS NATIONALITY,
  CASE
    WHEN PRR.description IN ('Not active', 'BT - Indiscipline', 'BT - Incompetency', 'BT - Alcohol issues') THEN 'BT'
    WHEN PRR.description IN ('UT - retirement', 'UT - shore job', 'Deceased', 'UT - medical reasons') THEN 'UT'
    WHEN PRR.description IN ('Joined elsewhere', 'Seafarer joined elsewhere - wants to join back',
      'Seafarer wants to restart sailing after a break',
      'Seafarer briefed with regard to his shortcoming and re-inducted',
      'Seafarer joined elsewhere for promotions and wants to join back',
      'Time Down') THEN 'Joined elsewhere'
  END AS INACTIVE_REASON,
  SE.sign_off_date AS SIGN_OFF_DATE,
  R.name AS RANK_NAME,
  COALESCE(SE.vessel_name, 'Others') AS VESSEL_NAME,
  COALESCE(UPPER(COMP.name), 'Other') AS DOC,
  VCAT.name AS VESSEL_CATEGORY,
  CAST(SE.imo_number AS LONG) AS IMO_NUMBER,
  CASE
    WHEN UPPER(VCAT.name) IN ('BULK CARRIER','CONTAINER','GEN CARGO / MULTI-PURPOSE VESSEL',
      'CAR CARRIER / RO-RO','CEMENT CARRIER','LOG CARRIER','REEFER CARGO',
      'HEAVY LIFT/PROJECT CARGO','WOODCHIP CARRIER','OBO CARRIER') THEN 'DRY'
    WHEN UPPER(VCAT.name) IN ('OIL TANKER','CHEM/OIL PROD TANKER','ASPHALT / BITUMEN TANKER',
      'LPG CARRIER (REFRI)','CHEMICAL TANKER','LNG CARRIER','LPG CARRIER (PRESS)',
      'SUPPLY /OFFSHORE / TUG BOAT / AHTS','GAS TANKER','OIL/PROD BUNKER BARGE','CHEM/PROD TANKER') THEN 'WET'
    ELSE NULL
  END AS VESSEL_FLEET_TYPE
FROM (
  -- Latest DEACTIVATION remark per seafarer (mirrors SAC: latest INACTIVE remark)
  SELECT *, ROW_NUMBER() OVER (PARTITION BY seafarer_id ORDER BY updated_at DESC) AS rn
  FROM curated_db.db_smac_prod_navitasai_crewing_shore.seafarer_remarks
  WHERE deleted_at IS NULL
    AND profile_remark_type_id = '4bf24d17-381a-429b-83aa-849cbf5279d6' -- DEACTIVATION type
) SR
INNER JOIN curated_db.db_smac_prod_navitasai_crewing_public.seafarers S
  ON S.id = SR.seafarer_id AND S.deleted_at IS NULL
LEFT JOIN curated_db.db_smac_prod_navitasai_masters_crewing.profile_remark_reasons PRR
  ON PRR.id = SR.profile_remark_reason_id
LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.nationalities N
  ON N.id = S.nationality_id
-- Latest sea experience for sign-off + vessel info
LEFT JOIN (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY seafarer_id ORDER BY COALESCE(sign_off_date, sign_on_date) DESC) AS rn
  FROM curated_db.db_smac_prod_navitasai_crewing_public.seafarer_sea_experiences
  WHERE deleted_at IS NULL
) SE ON SE.seafarer_id = S.id AND SE.rn = 1
LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.ranks R ON R.id = SE.rank_id
LEFT JOIN curated_db.db_smac_prod_navitasai_masters_vessel.categories VCAT ON VCAT.id = SE.vessel_category_id
LEFT JOIN curated_db.db_smac_prod_navitasai_masters_public.companies COMP ON COMP.id = SE.doc_holder_company_id
WHERE SR.rn = 1
  AND S.profile_status_id = '01993624-9d27-7fa6-8387-ba5a60c6b128'; -- Currently INACTIVE